# FACET (K=4 by age): Jupyter Notebook Version with Local Caches

This notebook is the **local Jupyter version** of a **FACET** multi-group experiment for the `Uncertainty_Relations` project.

It adapts the previous **MultiNLI** notebook to the **FACET** image-classification setting, with:

- **label** = occupation class (**20 classes**)
- **group** = perceived **age** (**4 groups**: Younger, Middle, Older, Unknown)
- selectable conformal score: **simple**, **SAPS**, or **RAPS**
- a **zero-shot CLIP ViT-L/14** base model that produces class probabilities

As before, the notebook keeps **outputs and caches inside the notebook directory**, which is convenient for local experiments, backup, and moving the whole project folder between machines.

## What this notebook does

This notebook loads **FACET** from a local directory, extracts **zero-shot CLIP** probabilities, and then runs the same three policy comparisons as in the BioBias and MultiNLI notebooks:

- **A. pooled threshold** -> group-wise coverage distortion  
- **B. group-wise thresholds** -> set-size heterogeneity  
- **C. equalized expected set size** -> coverage distortion

The implementation supports three nonconformity scores through a single switch:

- `CONFORMAL_SCORE = "simple"`
- `CONFORMAL_SCORE = "saps"`
- `CONFORMAL_SCORE = "raps"`

### Important FACET-specific note

This notebook follows the **FACET evaluation protocol** used in the conformal fairness literature:

- keep the **top 20 occupation classes**
- use **age** as the group variable with 4 groups
- filter to images with a **single person** and a **single occupation label**
- split the filtered dataset into:
  - `calibration_validation` of size **1400**
  - `calibration` of size **4000**
  - `test` of size **1400**
- stratify by **occupation label only**, while letting the age-group proportions remain natural

There is **no task-specific model training** here: the base model is **zero-shot CLIP**, and the notebook focuses on the post-hoc conformal analysis.

## Local project root and cache layout

This cell sets the project root to the **current notebook working directory** and redirects common caches into subfolders of that directory.

By default it creates:

- `./outputs`
- `./cache/huggingface`
- `./cache/huggingface/datasets`
- `./cache/huggingface/hub`
- `./cache/transformers`
- `./cache/torch`
- `./cache/matplotlib`
- `./cache/pip`
- `./cache/wandb`

In a normal local Jupyter workflow, `Path.cwd()` is the folder where the `.ipynb` lives.  
If your Jupyter server starts from another directory, change `PROJECT_ROOT` manually below.


In [ ]:
import os
from pathlib import Path

# ------------------------------------------------------------------
# Set the local project root.
# In a normal local Jupyter workflow, this is the notebook directory.
# If needed, replace Path.cwd() with a manual path.
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

# Central output directory
ROOT_DIR = PROJECT_ROOT / "outputs"

# Central cache directory
CACHE_ROOT = PROJECT_ROOT / "cache"
HF_HOME_DIR = CACHE_ROOT / "huggingface"
HF_DATASETS_CACHE_DIR = HF_HOME_DIR / "datasets"
HF_HUB_CACHE_DIR = HF_HOME_DIR / "hub"
TORCH_HOME_DIR = CACHE_ROOT / "torch"
MPLCONFIGDIR_DIR = CACHE_ROOT / "matplotlib"
PIP_CACHE_DIR = CACHE_ROOT / "pip"
WANDB_DIR = CACHE_ROOT / "wandb"
XDG_CACHE_HOME_DIR = CACHE_ROOT / "xdg"
TMP_DIR = CACHE_ROOT / "tmp"

for p in [
    ROOT_DIR,
    CACHE_ROOT,
    HF_HOME_DIR,
    HF_DATASETS_CACHE_DIR,
    HF_HUB_CACHE_DIR,
    TORCH_HOME_DIR,
    MPLCONFIGDIR_DIR,
    PIP_CACHE_DIR,
    WANDB_DIR,
    XDG_CACHE_HOME_DIR,
    TMP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# Redirect caches before importing heavy libraries.
# Prefer HF_HOME-based routing. Do not set TRANSFORMERS_CACHE because
# current Transformers warns that it is deprecated.
os.environ["HF_HOME"] = str(HF_HOME_DIR)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR_DIR)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
os.environ["WANDB_DIR"] = str(WANDB_DIR)
os.environ["WANDB_CACHE_DIR"] = str(WANDB_DIR / "cache")
os.environ["XDG_CACHE_HOME"] = str(XDG_CACHE_HOME_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ROOT_DIR:", ROOT_DIR)
print("CACHE_ROOT:", CACHE_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])
print("TORCH_HOME:", os.environ["TORCH_HOME"])
print("MPLCONFIGDIR:", os.environ["MPLCONFIGDIR"])
print("PIP_CACHE_DIR:", os.environ["PIP_CACHE_DIR"])
print("WANDB_DIR:", os.environ["WANDB_DIR"])
print("TMPDIR:", os.environ["TMPDIR"])
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])



## Local package behavior

This Jupyter version is intentionally conservative:

- It **does not auto-upgrade packages by default**.
- It prints a version report first.
- If something is missing, you can install it manually in the terminal for the current conda environment.

This avoids notebook-side `pip` failures in partially inconsistent local environments.



In [ ]:
from pathlib import Path

ROOT_DIR = Path(ROOT_DIR)
ROOT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT_DIR:", ROOT_DIR)
print("Outputs will be stored under:", ROOT_DIR)


## Configuration

The defaults below are set to a practical **FACET K=4** configuration:

- fixed primary alpha
- robustness alpha sweep
- one controlled group-temperature sweep
- **zero-shot CLIP ViT-L/14** inference
- selectable score: `simple`, `saps`, or `raps`

### Important local-data note

FACET is not loaded from HuggingFace here.  
Instead, the notebook expects a **local FACET folder** such as:

- `./data/facet/images/...`
- `./data/facet/annotations/...`

or the spelling used in some repos:

- `./data/facet/annotaions/...`

The metadata-file and column detection is written to be **fairly robust**, but because local FACET exports can vary, you may still need to adjust a few column overrides in the config cell after the first inspection printout.

If you first want a cheap smoke test, change:

- `EXPERIMENT_MODE = "quick"`
- `SEEDS = [0]`
- optionally reduce `QUICK_LIMITS`
- optionally set `RUN_TEMPERATURE_SWEEP = False`

Then switch back to `EXPERIMENT_MODE = "full"` for the main run.

In [ ]:
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple, set)):
        return [to_jsonable(x) for x in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, Path):
        return str(obj)
    else:
        return obj


# ============================================================
# Core experiment switches
# ============================================================
RUN_MODE = "train_and_analyze"   # "train_and_analyze" or "analyze_only"
EXPERIMENT_MODE = "full"         # "quick" or "full"
SEEDS = [4]

# ============================================================
# Conformal score / nonconformity score selection
# Choose one of: "simple" (1 - p), "raps", "saps"
# ============================================================
CONFORMAL_SCORE = "raps"   # "simple", "raps", or "saps"

SCORE_TEMPERATURE = 0.60      # temperature used *inside* the score (1.0 = no change)
SCORE_RANDOMIZE = False      # True -> u ~ Unif[0,1] per example; False -> u=0.5
SCORE_RANDOM_SEED = 0        # only used when SCORE_RANDOMIZE=True

# RAPS hyperparameters
RAPS_LAMBDA = 0.02
RAPS_K_REG = 1

# SAPS hyperparameters
SAPS_LAMBDA = 0.30


# ============================================================
# Dataset / model
# ============================================================
DATASET_NAME = "FACET"
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"
EXPECTED_NUM_GROUPS = None  # resolved after FACET_GROUP_VARIANT is applied
EXPECTED_NUM_LABELS = 20

# ------------------------------------------------------------------
# FACET local-file assumptions for YOUR current setup:
#   notebook folder/
#       facet_fixed.ipynb
#       annotations.zip
#       imgs_1/
#       imgs_2/
#       imgs_3/
# ------------------------------------------------------------------
FACET_ROOT = PROJECT_ROOT
FACET_METADATA_FILE = None  # leave None for autodiscovery; or set explicitly

FACET_IMAGE_SEARCH_DIRS = [
    PROJECT_ROOT / "imgs_1",
    PROJECT_ROOT / "imgs_2",
    PROJECT_ROOT / "imgs_3",
    PROJECT_ROOT / "images",
    PROJECT_ROOT / "imgs",
    PROJECT_ROOT / "data" / "facet" / "imgs_1",
    PROJECT_ROOT / "data" / "facet" / "imgs_2",
    PROJECT_ROOT / "data" / "facet" / "imgs_3",
    PROJECT_ROOT / "data" / "facet" / "images",
    PROJECT_ROOT / "data" / "facet" / "imgs",
]

FACET_METADATA_CANDIDATES = [
    PROJECT_ROOT / "annotations.zip",
    PROJECT_ROOT / "annotations" / "annotations.csv",
    PROJECT_ROOT / "annotations.csv",
    PROJECT_ROOT / "data" / "facet" / "annotations.zip",
    PROJECT_ROOT / "data" / "facet" / "annotations" / "annotations.csv",
]

# Optional explicit column overrides. Leave as None for autodetection.
FACET_COLUMN_OVERRIDES = {
    "image_path": None,
    "occupation": None,
    "secondary_occupation": None,
    "age_group": None,
    "num_people": None,
    "bbox": None,
    "age_young": None,
    "age_middle": None,
    "age_older": None,
    "age_unknown": None,
}

# ------------------------------------------------------------------
# Preprocessing upgrades for paper-grade FACET experiments
# ------------------------------------------------------------------

# Group variant:
# - "age4"            : Younger / Middle / Older / Unknown  (paper-aligned default)
# - "age3_no_unknown" : drop Unknown and keep only Younger / Middle / Older
FACET_GROUP_VARIANT = "age4"

# Optional score-temperature tuning on the calibration-validation split.
# This does NOT retrain CLIP. It chooses a global probability temperature
# that tends to keep pooled coverage near nominal while shortening sets.
AUTO_TUNE_SCORE_TEMPERATURE = False
AUTO_TUNE_PRIMARY_ONLY = True
AUTO_TUNE_TEMPERATURE_GRID = [0.50, 0.60, 0.70, 0.85, 1.00]
AUTO_TUNE_COVERAGE_TOL = 0.01

# Split-size plan:
# - "published" : use the published 1400 / 4000 / 1400 FACET split
# - "large_auto": use the same 20.6% / 58.8% / 20.6% proportions, but scaled
#                 up to the full filtered dataset for stronger statistical stability
FACET_SPLIT_SIZE_MODE = "large_auto"

# Fixed label list for reproducibility across score choices / seeds / crop modes.
# This list was obtained from the current annotations file by taking the 20 most
# frequent single-label occupations before any age-margin filtering.
FACET_LABEL_SELECTION_MODE = "fixed_list"
FACET_FIXED_LABELS = [
    "lawman",
    "laborer",
    "boatman",
    "tennis_player",
    "basketball_player",
    "backpacker",
    "speaker",
    "dancer",
    "motorcyclist",
    "farmer",
    "ballplayer",
    "soldier",
    "guard",
    "soccer_player",
    "singer",
    "repairman",
    "computer_user",
    "seller",
    "guitarist",
    "skateboarder",
]

# Stratification mode for split construction:
# - "label_only"  : match the older notebook behavior
# - "label_x_age" : the recommended default for this paper
FACET_SPLIT_STRATIFY_MODE = "label_x_age"

# Age-vote confidence filter. Keep rows only when (top vote - second vote) >= threshold.
# Set to 0 to disable. In the current FACET annotations the vote columns are mostly one-hot,
# so threshold 1 removes only tied/ambiguous rows.
FACET_AGE_MARGIN_MIN = 1.0

# Image mode:
# - "bbox_crop": crop the annotated person box before CLIP inference (recommended)
# - "full_image": use the entire image
FACET_IMAGE_MODE = "bbox_crop"
FACET_BBOX_EXPAND_RATIO = 0.08     # enlarge crop by 8% on each side
FACET_BBOX_MIN_SIDE_PX = 4         # reject degenerate boxes
FACET_DROP_AMBIGUOUS_IMAGE_MATCHES = True

# Prompting for zero-shot CLIP
CLIP_PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a portrait of a {}.",
    "a person working as a {}.",
]
CLIP_LABEL_ARTICLE = False

# Inference settings
INFERENCE_BATCH_SIZE_QUICK = 16
INFERENCE_BATCH_SIZE_FULL = 32
NUM_WORKERS = 0
USE_FP16 = False

# Optional override for re-using an existing output folder.
RUN_NAME_OVERRIDE = None

# 4-color palette for age groups
GROUP_COLOR_PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52"
]

# ============================================================
# Analysis layout
# ============================================================
PRIMARY_ALPHA = 0.10
ROBUSTNESS_ALPHAS = [0.05, 0.07, 0.085, 0.10]
RUN_ALPHA_ROBUSTNESS = True
RUN_TEMPERATURE_SWEEP = False

# Controlled heterogeneity sweep:
# keep one reference group at temperature 1.0 and scale the chosen group.
TEMPERATURE_SWEEP_GROUP = 0
TEMPERATURE_SWEEP_VALUES = [1.00, 1.10, 1.25, 1.50, 1.75, 2.00]
TEMPERATURE_EPS = 1e-12

GRID_STEP = 0.0001

# FACET target split sizes from the published setup
FACET_TARGET_SPLIT_SIZES_PUBLISHED = {
    "calibration_validation": 1400,
    "calibration": 4000,
    "test": 1400,
}

# Large-split robustness uses the same proportions as the published setup:
# 1400 / 6800, 4000 / 6800, 1400 / 6800.
FACET_AUTO_SPLIT_PROPORTIONS = {
    "calibration_validation": 1400 / 6800,
    "calibration": 4000 / 6800,
    "test": 1400 / 6800,
}

# Quick mode limits (applied after the seed-specific split)
QUICK_LIMITS = {
    "calibration_validation": 300,
    "calibration": 800,
    "test": 300,
}

# Full mode uses the published target sizes above, so no extra post-split subsampling is applied.
FULL_LIMITS = None

# Plot titles / file naming
RUN_TAG = f"facet_{CONFORMAL_SCORE}_{EXPERIMENT_MODE}"

# Simple text cleanup flags carried over from the MultiNLI notebook.
# They are not used in FACET image inference, but retained to avoid downstream edits.
APPLY_CLEAN_TEXT = False
TRUNCATE_TO_400_CHARS = False

ALPHAS = list(ROBUSTNESS_ALPHAS)
LIMITS = QUICK_LIMITS if EXPERIMENT_MODE == "quick" else FULL_LIMITS
INFERENCE_BATCH_SIZE = INFERENCE_BATCH_SIZE_QUICK if EXPERIMENT_MODE == "quick" else INFERENCE_BATCH_SIZE_FULL

if FACET_GROUP_VARIANT == "age4":
    FACET_SELECTED_AGE_GROUPS = ["Younger", "Middle", "Older", "Unknown"]
elif FACET_GROUP_VARIANT == "age3_no_unknown":
    FACET_SELECTED_AGE_GROUPS = ["Younger", "Middle", "Older"]
else:
    raise ValueError(f"Unknown FACET_GROUP_VARIANT={FACET_GROUP_VARIANT!r}")

EXPECTED_NUM_GROUPS = len(FACET_SELECTED_AGE_GROUPS)


# ============================================================
# Run directory
# ============================================================
_RUN_TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

# If you want to re-use an existing folder, set RUN_NAME_OVERRIDE above.
# Otherwise a fresh output folder is created automatically.
RUN_NAME = RUN_NAME_OVERRIDE or f"{RUN_TAG}_{_RUN_TIMESTAMP}"
RUN_ROOT = ROOT_DIR / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("RUN_NAME               =", RUN_NAME)
print("RUN_ROOT               =", RUN_ROOT)

print("Configuration summary")
print("-" * 80)
print("RUN_MODE               =", RUN_MODE)
print("EXPERIMENT_MODE        =", EXPERIMENT_MODE)
print("SEEDS                  =", SEEDS)
print("CONFORMAL_SCORE        =", CONFORMAL_SCORE)
print("CLIP_MODEL_NAME        =", CLIP_MODEL_NAME)
print("FACET_ROOT             =", FACET_ROOT)
print("FACET_METADATA_FILE    =", FACET_METADATA_FILE)
print("FACET_LABEL_MODE       =", FACET_LABEL_SELECTION_MODE)
print("FACET_GROUP_VARIANT    =", FACET_GROUP_VARIANT)
print("FACET_SELECTED_GROUPS  =", FACET_SELECTED_AGE_GROUPS)
print("FACET_SPLIT_SIZE_MODE  =", FACET_SPLIT_SIZE_MODE)
print("AUTO_TUNE_SCORE_TEMP   =", AUTO_TUNE_SCORE_TEMPERATURE)
print("FACET_FIXED_LABELS     =", FACET_FIXED_LABELS)
print("FACET_SPLIT_STRATIFY   =", FACET_SPLIT_STRATIFY_MODE)
print("FACET_AGE_MARGIN_MIN   =", FACET_AGE_MARGIN_MIN)
print("FACET_IMAGE_MODE       =", FACET_IMAGE_MODE)
print("FACET_TARGET_SPLITS    =", FACET_TARGET_SPLIT_SIZES_PUBLISHED)
print("FACET_BBOX_EXPAND_RATIO=", FACET_BBOX_EXPAND_RATIO)
print("PRIMARY_ALPHA          =", PRIMARY_ALPHA)
print("ALPHAS                 =", ALPHAS)
print("RUN_TEMPERATURE_SWEEP  =", RUN_TEMPERATURE_SWEEP)
print("TEMPERATURE_SWEEP_GROUP=", TEMPERATURE_SWEEP_GROUP)
print("TEMPERATURE_SWEEP_VALUES=", TEMPERATURE_SWEEP_VALUES)
print("INFERENCE_BATCH_SIZE   =", INFERENCE_BATCH_SIZE)

**Run note.** This FACET notebook defaults to a cautious first pass:

- `EXPERIMENT_MODE="quick"`
- `INFERENCE_BATCH_SIZE=16`
- `RUN_TEMPERATURE_SWEEP=False`

After the first successful run, switch to `EXPERIMENT_MODE="full"`, inspect the printed age-group mapping, choose the desired `TEMPERATURE_SWEEP_GROUP`, and then turn `RUN_TEMPERATURE_SWEEP=True`.
- `GRID_STEP=1e-4` to avoid coarse equalized-size threshold resolution when RAPS quantiles cluster near 1.


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_global_seed(SEEDS[0])


## Load and preprocess FACET

This notebook uses:

- **image** as the input
- **occupation** as the multiclass prediction target
- **perceived age** as the group variable

The goal is a **K = 4** real-data multi-group conformal experiment.

To keep the rest of the notebook minimally changed, we keep a few legacy variable names:

- `selected_profession_names` is reused as the list of label names
- `gender_names` is reused as the list of group names

Those legacy names are only compatibility aliases for the downstream plotting / analysis cells.

In [ ]:
import ast
import json
import zipfile
from datasets import Dataset

FACET_ALL_AGE_GROUPS = ["Younger", "Middle", "Older", "Unknown"]
FACET_AGE_GROUPS = list(FACET_SELECTED_AGE_GROUPS)
age_group_to_id = {name: idx for idx, name in enumerate(FACET_AGE_GROUPS)}


def _existing_dir_or_none(path_like):
    p = Path(path_like)
    return p if p.exists() else None


def _existing_file_or_none(path_like):
    p = Path(path_like)
    return p if p.exists() and p.is_file() else None


def _candidate_image_dirs():
    dirs = []
    for p in FACET_IMAGE_SEARCH_DIRS:
        p = _existing_dir_or_none(p)
        if p is not None and p not in dirs:
            dirs.append(p)
    for pattern in ["imgs*", "images*", "*imgs*", "*images*"]:
        for p in sorted(PROJECT_ROOT.glob(pattern)):
            if p.is_dir() and p not in dirs:
                dirs.append(p)
    return dirs


def autodiscover_facet_metadata_file():
    if FACET_METADATA_FILE not in [None, ""]:
        p = Path(FACET_METADATA_FILE)
        if not p.exists():
            raise FileNotFoundError(f"FACET_METADATA_FILE was set explicitly but does not exist: {p}")
        return p

    for cand in FACET_METADATA_CANDIDATES:
        p = _existing_file_or_none(cand)
        if p is not None:
            return p

    patterns = [
        "annotations.zip",
        "*annotations*.csv",
        "*annot*.csv",
        "*.csv",
        "*.parquet",
        "*.jsonl",
        "*.json",
    ]
    for pattern in patterns:
        for p in sorted(PROJECT_ROOT.rglob(pattern)):
            if p.is_file():
                return p

    raise FileNotFoundError(
        "Could not autodiscover a FACET metadata file. "
        "Put annotations.zip next to the notebook, or set FACET_METADATA_FILE explicitly."
    )


def load_table_any(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if isinstance(raw, list):
            return pd.DataFrame(raw)
        if isinstance(raw, dict):
            for key in ["data", "annotations", "items", "rows"]:
                if key in raw and isinstance(raw[key], list):
                    return pd.DataFrame(raw[key])
            return pd.DataFrame(raw)
    if suffix == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            names = zf.namelist()
            csv_candidates = [n for n in names if n.lower().endswith(".csv")]
            if len(csv_candidates) == 0:
                raise ValueError(f"Zip file has no CSV metadata inside: {path}")
            preferred = None
            for n in csv_candidates:
                lower = n.lower()
                if lower.endswith("annotations.csv"):
                    preferred = n
                    break
            if preferred is None:
                preferred = csv_candidates[0]
            print(f"Reading metadata from zip member: {preferred}")
            with zf.open(preferred) as f:
                return pd.read_csv(f)
    raise ValueError(f"Unsupported metadata-file format: {path}")


def normalize_column_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def choose_column(df: pd.DataFrame, kind: str, candidates, required: bool = True):
    override = FACET_COLUMN_OVERRIDES.get(kind, None)
    if override not in [None, ""]:
        if override not in df.columns:
            raise KeyError(f"FACET_COLUMN_OVERRIDES['{kind}']={override!r} not found in metadata columns.")
        return override

    normalized = {normalize_column_name(c): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        nc = normalize_column_name(cand)
        if nc in normalized:
            return normalized[nc]
    if required:
        raise KeyError(
            f"Could not detect a column for '{kind}'. "
            f"Available columns are: {list(df.columns)}. "
            "Set FACET_COLUMN_OVERRIDES in the config cell."
        )
    return None


def choose_age_vote_columns(df: pd.DataFrame):
    direct_age_col = choose_column(
        df,
        "age_group",
        candidates=[
            "age", "age_group", "perceived_age", "agebucket", "age_bucket",
            "agecategory", "age_category",
        ],
        required=False,
    )
    if direct_age_col is not None:
        return {"direct": direct_age_col}

    younger = choose_column(df, "age_young", ["age_presentation_young", "age_young", "young_votes"], required=False)
    middle = choose_column(df, "age_middle", ["age_presentation_middle", "age_middle", "middle_votes"], required=False)
    older = choose_column(df, "age_older", ["age_presentation_older", "age_older", "older_votes"], required=False)
    unknown = choose_column(df, "age_unknown", ["age_presentation_na", "age_unknown", "unknown_votes", "age_na"], required=False)

    cols = {"Younger": younger, "Middle": middle, "Older": older, "Unknown": unknown}
    if all(v is None for v in cols.values()):
        raise KeyError(
            "Could not detect FACET age columns. Expected either a direct age-group column, "
            "or vote columns such as age_presentation_young/middle/older/na."
        )
    return cols


def maybe_literal_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                val = ast.literal_eval(s)
                if isinstance(val, list):
                    return val
                if isinstance(val, tuple):
                    return list(val)
            except Exception:
                pass
    return [x]


def singleton_string(x):
    vals = maybe_literal_list(x)
    vals = [v for v in vals if str(v).strip() != "" and str(v).strip().lower() != "nan"]
    if len(vals) != 1:
        return None
    return str(vals[0]).strip()


def is_missing_or_none(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "nan", "none", "null", "na", "n/a"}


def canonicalize_occupation_raw(x):
    s = singleton_string(x)
    if s is None:
        return None
    s = s.strip().lower().replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s).strip("_")
    return s if s != "" else None


def occupation_display_name(raw_label: str) -> str:
    raw = str(raw_label).strip().lower()
    manual = {
        "computer_user": "Computer User",
        "tennis_player": "Tennis Player",
        "basketball_player": "Basketball Player",
        "soccer_player": "Soccer Player",
        "horseman": "Horseman",
        "lawman": "Lawman",
        "boatman": "Boatman",
        "ballplayer": "Ballplayer",
    }
    if raw in manual:
        return manual[raw]
    pretty = raw.replace("_", " ").replace("-", " ")
    pretty = re.sub(r"\s+", " ", pretty).strip()
    return pretty.title()


def canonicalize_age_group_direct(x):
    s = singleton_string(x)
    if s is None:
        return "Unknown"
    key = re.sub(r"\s+", " ", s.strip().lower())
    if key in {"younger", "young", "under25", "<25", "lt25"}:
        return "Younger"
    if key in {"middle", "middleaged", "25-65", "25to65", "adult"}:
        return "Middle"
    if key in {"older", "old", "65+", ">65", "senior"}:
        return "Older"
    if key in {"unknown", "unspecified", "na", "n/a", "none"}:
        return "Unknown"
    titled = key.title()
    if titled in FACET_AGE_GROUPS:
        return titled
    return "Unknown"


def age_vote_summary(row, vote_cols):
    if "direct" in vote_cols:
        group_name = canonicalize_age_group_direct(row[vote_cols["direct"]])
        return {
            "group_name": group_name,
            "age_vote_margin": np.nan,
            "age_top_vote": np.nan,
            "age_second_vote": np.nan,
            "age_vote_tie": False,
        }

    scores = []
    for group_name in FACET_ALL_AGE_GROUPS:
        col = vote_cols.get(group_name, None)
        if col is None:
            val = 0.0
        else:
            try:
                val = float(row[col])
            except Exception:
                val = 0.0
            if np.isnan(val):
                val = 0.0
        scores.append(val)

    scores = np.asarray(scores, dtype=float)
    if scores.max(initial=0.0) <= 0:
        return {
            "group_name": "Unknown",
            "age_vote_margin": 0.0,
            "age_top_vote": 0.0,
            "age_second_vote": 0.0,
            "age_vote_tie": True,
        }

    order = np.argsort(scores)[::-1]
    top_idx = int(order[0])
    top_vote = float(scores[top_idx])
    second_vote = float(scores[order[1]]) if len(order) > 1 else 0.0
    margin = float(top_vote - second_vote)
    return {
        "group_name": FACET_ALL_AGE_GROUPS[top_idx],
        "age_vote_margin": margin,
        "age_top_vote": top_vote,
        "age_second_vote": second_vote,
        "age_vote_tie": bool(margin <= 0),
    }


def coerce_num_people(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    vals = maybe_literal_list(x)
    if len(vals) == 1:
        try:
            return int(vals[0])
        except Exception:
            pass
    try:
        return int(x)
    except Exception:
        return np.nan


def parse_bbox_value(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    obj = x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null"}:
            return None
        try:
            obj = json.loads(s)
        except Exception:
            try:
                obj = ast.literal_eval(s)
            except Exception:
                return None
    if isinstance(obj, dict):
        if all(k in obj for k in ["x", "y", "width", "height"]):
            try:
                x0 = float(obj["x"])
                y0 = float(obj["y"])
                w = float(obj["width"])
                h = float(obj["height"])
            except Exception:
                return None
            return {"x": x0, "y": y0, "width": w, "height": h}
        return None
    if isinstance(obj, (list, tuple)) and len(obj) >= 4:
        try:
            x0, y0, w, h = map(float, obj[:4])
        except Exception:
            return None
        return {"x": x0, "y": y0, "width": w, "height": h}
    return None


_IMAGE_FILENAME_INDEX = None
_AMBIGUOUS_IMAGE_BASENAMES = set()


def build_image_filename_index():
    global _IMAGE_FILENAME_INDEX, _AMBIGUOUS_IMAGE_BASENAMES
    if _IMAGE_FILENAME_INDEX is not None:
        return _IMAGE_FILENAME_INDEX

    index = {}
    candidate_dirs = _candidate_image_dirs()
    print("Candidate image roots:")
    for d in candidate_dirs:
        print("  -", d)

    for root in candidate_dirs:
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
                continue
            index.setdefault(p.name, []).append(p)

    _IMAGE_FILENAME_INDEX = index
    _AMBIGUOUS_IMAGE_BASENAMES = {name for name, matches in index.items() if len(matches) > 1}
    print(f"Indexed {sum(len(v) for v in index.values())} image files across {len(candidate_dirs)} roots.")
    print(f"Ambiguous basenames across roots: {len(_AMBIGUOUS_IMAGE_BASENAMES)}")
    return _IMAGE_FILENAME_INDEX


def resolve_image_path(x):
    s = singleton_string(x)
    if s is None:
        return None
    p = Path(s)

    if p.is_absolute() and p.exists():
        return str(p.resolve())

    candidate_dirs = _candidate_image_dirs()
    if not p.is_absolute():
        for root in candidate_dirs + [PROJECT_ROOT, FACET_ROOT]:
            cand = root / p
            if cand.exists():
                return str(cand.resolve())

    basename = p.name
    if basename == "":
        return None

    index = build_image_filename_index()
    matches = index.get(basename, [])
    if len(matches) == 1:
        return str(matches[0].resolve())
    if len(matches) > 1:
        if FACET_DROP_AMBIGUOUS_IMAGE_MATCHES:
            return None
        best = sorted(matches, key=lambda z: (len(z.parts), str(z)))[0]
        return str(best.resolve())
    return None


def label_manifest_from_counts(counts: pd.Series) -> pd.DataFrame:
    frame = counts.reset_index()
    frame.columns = ["occupation_raw", "count"]
    frame["occupation_display"] = frame["occupation_raw"].map(occupation_display_name)
    frame["label_id"] = frame["occupation_raw"].map(lambda x: selected_raw_labels.index(x) if x in selected_raw_labels else -1)
    return frame[["label_id", "occupation_raw", "occupation_display", "count"]].sort_values(["label_id", "occupation_raw"])


METADATA_PATH = autodiscover_facet_metadata_file()
raw_meta = load_table_any(METADATA_PATH)

print("Detected FACET metadata file:", METADATA_PATH)
print("Metadata shape:", raw_meta.shape)
print("Metadata columns:")
print(list(raw_meta.columns))

image_col = choose_column(
    raw_meta,
    "image_path",
    candidates=[
        "image_path", "img_path", "path", "filepath", "file_path", "relative_path",
        "img", "image", "imagefile", "image_name", "filename", "file_name",
    ],
    required=True,
)
occupation_col = choose_column(
    raw_meta,
    "occupation",
    candidates=[
        "class1", "primary_class", "occupation", "occupation_label",
        "class_label", "class_name", "label", "fine_grained_class",
        "activity", "category",
    ],
    required=True,
)
secondary_occupation_col = choose_column(
    raw_meta,
    "secondary_occupation",
    candidates=["class2", "secondary_class", "occupation2", "label2", "secondary_label"],
    required=False,
)
num_people_col = choose_column(
    raw_meta,
    "num_people",
    candidates=[
        "num_people", "person_count", "n_people", "people_count", "num_persons",
        "persons", "num_subjects",
    ],
    required=False,
)
bbox_col = choose_column(
    raw_meta,
    "bbox",
    candidates=["bounding_box", "bbox", "person_bbox", "crop_box", "box"],
    required=False,
)
age_vote_cols = choose_age_vote_columns(raw_meta)

print("\nDetected columns:")
print("  image_path          :", image_col)
print("  occupation          :", occupation_col)
print("  secondary_occupation:", secondary_occupation_col)
print("  num_people          :", num_people_col)
print("  bbox                :", bbox_col)
print("  age columns         :", age_vote_cols)

facet_df = raw_meta.copy()
facet_df["image_basename"] = facet_df[image_col].map(lambda x: Path(singleton_string(x)).name if singleton_string(x) is not None else None)
facet_df["occupation_raw"] = facet_df[occupation_col].map(canonicalize_occupation_raw)
facet_df["has_single_label"] = True
if secondary_occupation_col is not None:
    facet_df["has_single_label"] = facet_df[secondary_occupation_col].map(is_missing_or_none)

age_summary_rows = facet_df.apply(lambda row: age_vote_summary(row, age_vote_cols), axis=1)
age_summary_df = pd.DataFrame(list(age_summary_rows))
facet_df = pd.concat([facet_df, age_summary_df], axis=1)

facet_df["image_path_resolved"] = facet_df[image_col].map(resolve_image_path)

if num_people_col is not None:
    facet_df["num_people_clean"] = facet_df[num_people_col].map(coerce_num_people)
else:
    facet_df["num_people_clean"] = np.nan

per_image_counts = facet_df["image_basename"].value_counts(dropna=False)
facet_df["persons_in_source_image"] = facet_df["image_basename"].map(per_image_counts).fillna(0).astype(int)

facet_df["bbox_raw"] = facet_df[bbox_col].map(parse_bbox_value) if bbox_col is not None else None
facet_df["bbox_x"] = facet_df["bbox_raw"].map(lambda d: float(d["x"]) if isinstance(d, dict) else np.nan)
facet_df["bbox_y"] = facet_df["bbox_raw"].map(lambda d: float(d["y"]) if isinstance(d, dict) else np.nan)
facet_df["bbox_w"] = facet_df["bbox_raw"].map(lambda d: float(d["width"]) if isinstance(d, dict) else np.nan)
facet_df["bbox_h"] = facet_df["bbox_raw"].map(lambda d: float(d["height"]) if isinstance(d, dict) else np.nan)
facet_df["bbox_valid"] = (
    facet_df["bbox_w"].fillna(-1) >= float(FACET_BBOX_MIN_SIDE_PX)
) & (
    facet_df["bbox_h"].fillna(-1) >= float(FACET_BBOX_MIN_SIDE_PX)
)

single_label_df = facet_df.loc[facet_df["has_single_label"]].copy()
class_counts = single_label_df["occupation_raw"].dropna().value_counts().sort_values(ascending=False)

print("\nTop 30 single-label occupations by frequency:")
display(class_counts.head(30).to_frame("count"))

if FACET_LABEL_SELECTION_MODE == "fixed_list":
    if FACET_FIXED_LABELS is None:
        raise ValueError("FACET_LABEL_SELECTION_MODE='fixed_list' but FACET_FIXED_LABELS is None.")
    selected_raw_labels = [canonicalize_occupation_raw(x) for x in FACET_FIXED_LABELS]
    selected_raw_labels = [x for x in selected_raw_labels if x is not None]
else:
    selected_raw_labels = class_counts.head(EXPECTED_NUM_LABELS).index.tolist()

if len(selected_raw_labels) != EXPECTED_NUM_LABELS:
    raise ValueError(
        f"Expected {EXPECTED_NUM_LABELS} selected labels, but found {len(selected_raw_labels)}."
    )

selected_profession_names = [occupation_display_name(x) for x in selected_raw_labels]
selected_profession_ids = list(range(len(selected_raw_labels)))
profession_names = list(selected_profession_names)
label_raw_to_id = {raw: idx for idx, raw in enumerate(selected_raw_labels)}
label_raw_to_display = {raw: disp for raw, disp in zip(selected_raw_labels, selected_profession_names)}

gender_names = list(FACET_AGE_GROUPS)
group_names = list(FACET_AGE_GROUPS)

base_mask = (
    facet_df["occupation_raw"].isin(selected_raw_labels)
    & facet_df["group_name"].isin(FACET_AGE_GROUPS)
    & facet_df["image_path_resolved"].notna()
    & facet_df["has_single_label"]
)

if FACET_AGE_MARGIN_MIN not in [None, "", False]:
    base_mask = base_mask & (facet_df["age_vote_margin"].fillna(np.inf) >= float(FACET_AGE_MARGIN_MIN))

if FACET_IMAGE_MODE == "full_image":
    # Full-image inference is only clean for single-person source images.
    base_mask = base_mask & (facet_df["persons_in_source_image"] == 1)
elif FACET_IMAGE_MODE == "bbox_crop":
    if bbox_col is None:
        raise ValueError("FACET_IMAGE_MODE='bbox_crop' requires a bbox column, but none was detected.")
    base_mask = base_mask & facet_df["bbox_valid"]
else:
    raise ValueError(f"Unknown FACET_IMAGE_MODE={FACET_IMAGE_MODE!r}")

filtered_cols = [
    "image_basename",
    "image_path_resolved",
    "occupation_raw",
    "group_name",
    "age_vote_margin",
    "persons_in_source_image",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
]
filtered_df = facet_df.loc[base_mask, filtered_cols].copy()
filtered_df = filtered_df.rename(columns={"image_path_resolved": "image_path"})
filtered_df["occupation_display"] = filtered_df["occupation_raw"].map(label_raw_to_display)
filtered_df["labels"] = filtered_df["occupation_raw"].map(label_raw_to_id)
filtered_df["group"] = filtered_df["group_name"].map(age_group_to_id)
filtered_df["use_bbox_crop"] = FACET_IMAGE_MODE == "bbox_crop"
filtered_df = filtered_df.reset_index(drop=True)

# ------------------------------------------------------------
# Preprocessing audit tables
# ------------------------------------------------------------
audit_dir = RUN_ROOT / "preprocessing_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

stage_rows = []
def _record_stage(stage_name, mask):
    mask = np.asarray(mask, dtype=bool)
    stage_rows.append({
        "stage": stage_name,
        "rows_kept": int(mask.sum()),
        "rows_dropped": int((~mask).sum()),
        "keep_rate": float(mask.mean()) if len(mask) > 0 else np.nan,
    })

mask_occ = facet_df["occupation_raw"].isin(selected_raw_labels)
mask_single = facet_df["has_single_label"]
mask_age = facet_df["group_name"].isin(FACET_SELECTED_AGE_GROUPS)
mask_age_margin = facet_df["age_vote_margin"].fillna(np.inf) >= float(FACET_AGE_MARGIN_MIN) if FACET_AGE_MARGIN_MIN not in [None, "", False] else np.ones(len(facet_df), dtype=bool)
mask_resolved = facet_df["image_path_resolved"].notna()
mask_bbox = facet_df["bbox_valid"] if FACET_IMAGE_MODE == "bbox_crop" else np.ones(len(facet_df), dtype=bool)
mask_full_image_single_person = (facet_df["persons_in_source_image"] == 1) if FACET_IMAGE_MODE == "full_image" else np.ones(len(facet_df), dtype=bool)

_record_stage("selected_top20_labels", mask_occ)
_record_stage("single_label", mask_occ & mask_single)
_record_stage("valid_age_group", mask_occ & mask_single & mask_age)
_record_stage("age_margin_pass", mask_occ & mask_single & mask_age & mask_age_margin)
_record_stage("image_path_resolved", mask_occ & mask_single & mask_age & mask_age_margin & mask_resolved)
if FACET_IMAGE_MODE == "bbox_crop":
    _record_stage("bbox_valid", mask_occ & mask_single & mask_age & mask_age_margin & mask_resolved & mask_bbox)
if FACET_IMAGE_MODE == "full_image":
    _record_stage("single_person_source_image", mask_occ & mask_single & mask_age & mask_age_margin & mask_resolved & mask_full_image_single_person)
_record_stage("final_kept", base_mask)

preprocess_audit_df = pd.DataFrame(stage_rows)
preprocess_audit_df.insert(0, "raw_total_rows", len(facet_df))
preprocess_audit_df.to_csv(audit_dir / "preprocessing_audit.csv", index=False)

selected_counts_full = (
    facet_df.loc[facet_df["occupation_raw"].isin(selected_raw_labels) & facet_df["has_single_label"], "occupation_raw"]
    .value_counts()
    .reindex(selected_raw_labels, fill_value=0)
)
label_manifest_df = label_manifest_from_counts(selected_counts_full)
label_manifest_df.to_csv(audit_dir / "label_manifest.csv", index=False)

group_counts_df = (
    filtered_df["group_name"]
    .value_counts()
    .rename_axis("group_name")
    .reset_index(name="count")
    .sort_values("group_name")
    .reset_index(drop=True)
)
group_counts_df.to_csv(audit_dir / "group_counts_filtered.csv", index=False)

label_counts_df = (
    filtered_df["occupation_display"]
    .value_counts()
    .rename_axis("occupation_display")
    .reset_index(name="count")
)
label_counts_df.to_csv(audit_dir / "label_counts_filtered.csv", index=False)

group_label_counts_df = (
    filtered_df.groupby(["group_name", "occupation_display"]).size().reset_index(name="count")
)
group_label_counts_df.to_csv(audit_dir / "group_label_counts_filtered.csv", index=False)

group_label_pivot = (
    group_label_counts_df.pivot(index="group_name", columns="occupation_display", values="count")
    .fillna(0)
    .astype(int)
)
group_label_pivot.to_csv(audit_dir / "group_label_matrix_filtered.csv")

filename_duplicate_summary_df = pd.DataFrame({
    "metric": [
        "rows_after_filtering",
        "unique_image_basenames_after_filtering",
        "rows_with_duplicate_basenames_after_filtering",
        "ambiguous_basenames_in_local_image_index",
    ],
    "value": [
        int(len(filtered_df)),
        int(filtered_df["image_basename"].nunique()),
        int(filtered_df["image_basename"].duplicated(keep=False).sum()),
        int(len(_AMBIGUOUS_IMAGE_BASENAMES)),
    ],
})
filename_duplicate_summary_df.to_csv(audit_dir / "filename_duplicate_summary.csv", index=False)

missing_images = int((facet_df["image_path_resolved"].isna()).sum())
print("\nFiltering summary:")
print("  raw rows                                :", len(facet_df))
print("  single-label rows                       :", int(facet_df["has_single_label"].sum()))
print("  unresolved image paths                  :", missing_images)
print("  image mode                              :", FACET_IMAGE_MODE)
print("  selected age groups                     :", FACET_SELECTED_AGE_GROUPS)
print("  age-margin threshold                    :", FACET_AGE_MARGIN_MIN)
print("  kept rows                               :", len(filtered_df))
print("  dropped rows                            :", len(facet_df) - len(filtered_df))
print("  duplicate filtered basenames            :", int(filtered_df["image_basename"].duplicated(keep=False).sum()))
print("  selected raw labels                     :", selected_raw_labels)
print("  selected display labels                 :", selected_profession_names)

print("\nSaved preprocessing audit tables to:", audit_dir)

print("\nPreprocessing audit:")
display(preprocess_audit_df)

print("\nSelected label manifest:")
display(label_manifest_df)

print("\nSelected occupation counts after filtering:")
display(label_counts_df)

print("\nAge-group counts after filtering:")
display(group_counts_df)

print("\nGroup x label count matrix (filtered):")
display(group_label_pivot)

if filtered_df["labels"].nunique() != EXPECTED_NUM_LABELS:
    raise ValueError(
        f"Expected {EXPECTED_NUM_LABELS} labels after filtering, "
        f"but found {filtered_df['labels'].nunique()}."
    )

if filtered_df["group"].nunique() != EXPECTED_NUM_GROUPS:
    raise ValueError(
        f"Expected {EXPECTED_NUM_GROUPS} groups after filtering, "
        f"but found {filtered_df['group'].nunique()}."
    )

prepared = Dataset.from_pandas(
    filtered_df[
        [
            "image_path",
            "labels",
            "group",
            "group_name",
            "image_basename",
            "age_vote_margin",
            "bbox_x",
            "bbox_y",
            "bbox_w",
            "bbox_h",
            "use_bbox_crop",
            "persons_in_source_image",
        ]
    ].copy(),
    preserve_index=False,
)

print("\nLegacy-compatible label aliases:", selected_profession_names)
print("Legacy-compatible group aliases:", gender_names)

print("\nAge-group id mapping:")
for gid, gname in enumerate(gender_names):
    print(f"  {gid}: {gname}")

if not (0 <= int(TEMPERATURE_SWEEP_GROUP) < len(gender_names)):
    raise ValueError(
        f"TEMPERATURE_SWEEP_GROUP={TEMPERATURE_SWEEP_GROUP} is out of range for "
        f"{len(gender_names)} groups. Choose an id from the mapping printed above."
    )
else:
    print(
        f"Current TEMPERATURE_SWEEP_GROUP={TEMPERATURE_SWEEP_GROUP} -> "
        f"{gender_names[int(TEMPERATURE_SWEEP_GROUP)]}"
    )

prepared

In [ ]:
# FACET preprocessing is already completed in the previous cell.
# This cell replaces a leftover MultiNLI block that referenced `raw`,
# `genre_to_id`, `premise`, and `hypothesis`.

print("FACET preprocessing already completed in the previous cell.")
print(f"Usable examples after FACET filtering: {len(prepared)}")
print(f"Number of labels: {len(selected_profession_names)}")
print(f"Number of groups: {len(gender_names)}")

label_counts = pd.Series(prepared["labels"]).value_counts().sort_index()
group_counts = pd.Series(prepared["group_name"]).value_counts().reindex(gender_names, fill_value=0)

print("\nLabel counts (by label id):")
display(label_counts.to_frame("count"))
print("\nGroup counts:")
display(group_counts.to_frame("count"))

prepared.select(range(min(5, len(prepared))))



## Split helpers

This notebook follows the FACET split plan used in the conformal-fairness papers:

- `calibration_validation` = 1400
- `calibration` = 4000
- `test` = 1400

Stratification is done by **occupation label** only.  
We intentionally do **not** rebalance age groups, so the group proportions remain close to the natural FACET distribution.

In [ ]:
from sklearn.model_selection import train_test_split



def resolve_target_split_sizes(n_total: int):
    mode = str(FACET_SPLIT_SIZE_MODE).strip().lower()
    if mode == "published":
        out = {k: int(v) for k, v in FACET_TARGET_SPLIT_SIZES_PUBLISHED.items()}
    elif mode == "large_auto":
        props = dict(FACET_AUTO_SPLIT_PROPORTIONS)
        raw = {k: float(props[k]) * float(n_total) for k in props}
        out = {k: int(math.floor(v)) for k, v in raw.items()}
        allocated = sum(out.values())
        order = sorted(raw.keys(), key=lambda k: (raw[k] - out[k]), reverse=True)
        i = 0
        while allocated < n_total and len(order) > 0:
            out[order[i % len(order)]] += 1
            allocated += 1
            i += 1
        # Keep exactly n_total rows; all filtered rows are used in large_auto mode.
        # A tiny final correction guards against numerical quirks.
        while sum(out.values()) > n_total:
            largest_key = max(out, key=out.get)
            out[largest_key] -= 1
    else:
        raise ValueError(f"Unknown FACET_SPLIT_SIZE_MODE={FACET_SPLIT_SIZE_MODE!r}")
    return out


def choose_split_stratify_or_none(labels, groups, mode=None):
    mode = (mode or FACET_SPLIT_STRATIFY_MODE).strip().lower()
    label_series = pd.Series(labels).astype(str)
    group_series = pd.Series(groups).astype(str)

    if mode == "label_only":
        counts = label_series.value_counts()
        if (counts < 2).any():
            return None
        return label_series

    if mode == "label_x_age":
        joint = label_series + "||" + group_series
        joint_counts = joint.value_counts()
        if (joint_counts >= 2).all():
            return joint

        # Fallback: rare joint cells revert to label-only while keeping the rest joint-stratified.
        rare_joint = joint.map(joint_counts) < 2
        hybrid = joint.where(~rare_joint, label_series)
        hybrid_counts = hybrid.value_counts()
        if (hybrid_counts >= 2).all():
            return hybrid

        label_counts = label_series.value_counts()
        if (label_counts >= 2).all():
            return label_series

        return None

    raise ValueError(f"Unknown FACET_SPLIT_STRATIFY_MODE={mode!r}")


def sample_n_stratified(indices, labels, groups, n_select, seed=0):
    indices = np.asarray(indices)
    labels = np.asarray(labels)
    groups = np.asarray(groups)
    if n_select > len(indices):
        raise ValueError(f"Cannot select n={n_select} from only {len(indices)} examples.")
    if n_select == len(indices):
        return np.sort(indices), np.array([], dtype=int)

    strata = choose_split_stratify_or_none(labels, groups, mode=FACET_SPLIT_STRATIFY_MODE)
    keep_idx, rest_idx = train_test_split(
        indices,
        train_size=n_select,
        random_state=seed,
        stratify=strata,
    )
    return np.sort(keep_idx), np.sort(rest_idx)


def stratified_subsample_indices(labels, groups, max_samples, seed=0):
    n = len(labels)
    idx = np.arange(n)
    if max_samples is None or max_samples >= n:
        return idx

    strata = choose_split_stratify_or_none(labels, groups, mode=FACET_SPLIT_STRATIFY_MODE)
    keep_idx, _ = train_test_split(
        idx,
        train_size=max_samples,
        random_state=seed,
        stratify=strata,
    )
    return np.sort(keep_idx)


def build_splits_for_seed(seed: int):
    labels_all = np.array(prepared["labels"])
    groups_all = np.array(prepared["group"])
    idx_all = np.arange(len(prepared))

    target_sizes = resolve_target_split_sizes(len(idx_all))
    n_calval = int(target_sizes["calibration_validation"])
    n_cal = int(target_sizes["calibration"])
    n_test = int(target_sizes["test"])
    n_required = n_calval + n_cal + n_test

    if len(idx_all) < n_required:
        raise ValueError(
            f"After FACET filtering there are only {len(idx_all)} usable rows, "
            f"but the requested split sizes need {n_required}."
        )

    test_idx, remaining = sample_n_stratified(
        idx_all,
        labels_all,
        groups_all,
        n_select=n_test,
        seed=seed + 200,
    )
    remaining_labels = labels_all[remaining]
    remaining_groups = groups_all[remaining]

    calval_idx, remaining2 = sample_n_stratified(
        remaining,
        remaining_labels,
        remaining_groups,
        n_select=n_calval,
        seed=seed + 100,
    )
    remaining2_labels = labels_all[remaining2]
    remaining2_groups = groups_all[remaining2]

    cal_idx, _unused = sample_n_stratified(
        remaining2,
        remaining2_labels,
        remaining2_groups,
        n_select=n_cal,
        seed=seed,
    )

    splits = {
        "calibration_validation": prepared.select(calval_idx.tolist()),
        "calibration": prepared.select(cal_idx.tolist()),
        "test": prepared.select(test_idx.tolist()),
    }

    for split_name, ds in list(splits.items()):
        labels = np.array(ds["labels"])
        groups = np.array(ds["group"])
        max_samples = None if LIMITS is None else LIMITS.get(split_name)
        keep_idx = stratified_subsample_indices(
            labels=labels,
            groups=groups,
            max_samples=max_samples,
            seed=seed,
        )
        splits[split_name] = ds.select(keep_idx.tolist())
    return splits


def summarize_split(ds, split_name):
    frame = pd.DataFrame({
        "label": ds["labels"],
        "group": ds["group"],
    })
    summary = frame.groupby(["group", "label"]).size().reset_index(name="count")
    summary["group_name"] = summary["group"].map(lambda x: gender_names[x] if x < len(gender_names) else str(x))
    summary["label_name"] = summary["label"].map(
        lambda x: selected_profession_names[x] if x < len(selected_profession_names) else str(x)
    )
    summary.insert(0, "split", split_name)
    return summary


preview_splits = build_splits_for_seed(SEEDS[0])
for split_name, ds in preview_splits.items():
    print(split_name, len(ds))

preview_summary = pd.concat(
    [summarize_split(ds, split_name) for split_name, ds in preview_splits.items()],
    ignore_index=True,
)
display(preview_summary.head(20))

## CLIP inference utilities

The base model in this notebook is **zero-shot CLIP ViT-L/14**.

For each FACET image, we construct class probabilities over the 20 retained occupations using prompt-based zero-shot classification. These probabilities are then passed into the same conformal-analysis pipeline used in the previous notebooks.

In [ ]:
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoProcessor, CLIPModel


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_CLIP_ARTIFACTS = None


def format_clip_label(label: str) -> str:
    if not CLIP_LABEL_ARTICLE:
        return label.lower()
    lowered = label.lower()
    article = "an" if lowered[:1] in {"a", "e", "i", "o", "u"} else "a"
    return f"{article} {lowered}"


def build_prompt_list(label: str):
    label_text = format_clip_label(label)
    return [template.format(label_text) for template in CLIP_PROMPT_TEMPLATES]


def _normalize_rows(x: torch.Tensor) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def get_clip_artifacts():
    global _CLIP_ARTIFACTS
    if _CLIP_ARTIFACTS is not None:
        return _CLIP_ARTIFACTS

    processor = AutoProcessor.from_pretrained(CLIP_MODEL_NAME)
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
    model.eval()
    model.to(DEVICE)

    text_feature_rows = []
    with torch.no_grad():
        for label in selected_profession_names:
            prompts = build_prompt_list(label)
            text_inputs = processor(
                text=prompts,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            text_inputs = {k: v.to(DEVICE) for k, v in text_inputs.items()}
            text_features = model.get_text_features(**text_inputs)
            text_features = _normalize_rows(text_features)
            mean_feature = text_features.mean(dim=0, keepdim=True)
            mean_feature = _normalize_rows(mean_feature)
            text_feature_rows.append(mean_feature.squeeze(0))

    text_feature_bank = torch.stack(text_feature_rows, dim=0)
    text_feature_bank = _normalize_rows(text_feature_bank)

    _CLIP_ARTIFACTS = {
        "processor": processor,
        "model": model,
        "text_feature_bank": text_feature_bank,
    }
    return _CLIP_ARTIFACTS


def _safe_bbox_crop(image: Image.Image, x: float, y: float, w: float, h: float, expand_ratio: float = 0.0) -> Image.Image:
    W, H = image.size
    x1 = float(x)
    y1 = float(y)
    x2 = float(x) + float(w)
    y2 = float(y) + float(h)

    if expand_ratio not in [None, 0, 0.0]:
        dx = float(w) * float(expand_ratio)
        dy = float(h) * float(expand_ratio)
        x1 -= dx
        x2 += dx
        y1 -= dy
        y2 += dy

    x1 = max(0, min(W - 1, int(np.floor(x1))))
    y1 = max(0, min(H - 1, int(np.floor(y1))))
    x2 = max(x1 + 1, min(W, int(np.ceil(x2))))
    y2 = max(y1 + 1, min(H, int(np.ceil(y2))))

    return image.crop((x1, y1, x2, y2))


class FacetImageDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.image_paths = list(hf_dataset["image_path"])
        self.labels = list(hf_dataset["labels"])
        self.use_bbox_crop = list(hf_dataset["use_bbox_crop"]) if "use_bbox_crop" in hf_dataset.column_names else [False] * len(self.image_paths)
        self.bbox_x = list(hf_dataset["bbox_x"]) if "bbox_x" in hf_dataset.column_names else [np.nan] * len(self.image_paths)
        self.bbox_y = list(hf_dataset["bbox_y"]) if "bbox_y" in hf_dataset.column_names else [np.nan] * len(self.image_paths)
        self.bbox_w = list(hf_dataset["bbox_w"]) if "bbox_w" in hf_dataset.column_names else [np.nan] * len(self.image_paths)
        self.bbox_h = list(hf_dataset["bbox_h"]) if "bbox_h" in hf_dataset.column_names else [np.nan] * len(self.image_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        if bool(self.use_bbox_crop[idx]):
            image = _safe_bbox_crop(
                image,
                x=float(self.bbox_x[idx]),
                y=float(self.bbox_y[idx]),
                w=float(self.bbox_w[idx]),
                h=float(self.bbox_h[idx]),
                expand_ratio=float(FACET_BBOX_EXPAND_RATIO),
            )
        label = int(self.labels[idx])
        return image, label


def make_facet_collate_fn(processor):
    def collate_fn(batch):
        images, labels = zip(*batch)
        model_inputs = processor(images=list(images), return_tensors="pt")
        labels = torch.tensor(labels, dtype=torch.long)
        return model_inputs, labels
    return collate_fn


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def predict_probabilities_zero_shot(ds):
    artifacts = get_clip_artifacts()
    processor = artifacts["processor"]
    model = artifacts["model"]
    text_feature_bank = artifacts["text_feature_bank"]

    torch_ds = FacetImageDataset(ds)
    loader = DataLoader(
        torch_ds,
        batch_size=INFERENCE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        collate_fn=make_facet_collate_fn(processor),
    )

    all_probs = []
    with torch.no_grad():
        for model_inputs, labels in loader:
            model_inputs = {k: v.to(DEVICE) for k, v in model_inputs.items()}
            image_features = model.get_image_features(**model_inputs)
            image_features = _normalize_rows(image_features)
            logits = model.logit_scale.exp() * image_features @ text_feature_bank.T
            probs = torch.softmax(logits, dim=-1)
            all_probs.append(probs.cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    labels = np.array(ds["labels"], dtype=int)
    return probs, labels


def compute_metrics_from_probs(probs: np.ndarray, labels: np.ndarray):
    preds = np.argmax(probs, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


def extract_probabilities_for_seed(seed: int, splits):
    set_global_seed(seed)

    probs_calval, y_calval = predict_probabilities_zero_shot(splits["calibration_validation"])
    probs_cal, y_cal = predict_probabilities_zero_shot(splits["calibration"])
    probs_test, y_test = predict_probabilities_zero_shot(splits["test"])

    metrics = {
        "calibration_validation": compute_metrics_from_probs(probs_calval, y_calval),
        "calibration": compute_metrics_from_probs(probs_cal, y_cal),
        "test": compute_metrics_from_probs(probs_test, y_test),
    }

    summary = {
        "seed": seed,
        "device": str(DEVICE),
        "clip_model_name": CLIP_MODEL_NAME,
        "prompt_templates": CLIP_PROMPT_TEMPLATES,
        "facet_image_mode": FACET_IMAGE_MODE,
        "facet_bbox_expand_ratio": FACET_BBOX_EXPAND_RATIO,
        "split_metrics": metrics,
    }

    arrays = {
        "probs_calibration_validation": probs_calval,
        "y_calibration_validation": y_calval,
        "g_calibration_validation": np.array(splits["calibration_validation"]["group"]),
        "probs_cal": probs_cal,
        "y_cal": y_cal,
        "g_cal": np.array(splits["calibration"]["group"]),
        "probs_test": probs_test,
        "y_test": y_test,
        "g_test": np.array(splits["test"]["group"]),
    }
    return summary, arrays

## Conformal utilities and plotting helpers

This notebook supports **selectable nonconformity scores** via `CONFORMAL_SCORE`:

- `CONFORMAL_SCORE = "simple"`:  
  standard score \(s(x,y) = 1 - \hat p_y(x)\)

- `CONFORMAL_SCORE = "raps"`:  
  regularized adaptive prediction sets (RAPS)

- `CONFORMAL_SCORE = "saps"`:  
  size-adaptive prediction sets (SAPS)

Everything downstream uses the same score switch, so you can reuse the same notebook for the three score families without changing the rest of the pipeline.

In [ ]:
# Generic plotting / conformal helpers (kept largely unchanged from the BioBias notebook)


from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})


def _standardize_group_name(name: str) -> str:
    raw = str(name).strip()
    lower = raw.lower()
    if lower == "male":
        return "Male"
    if lower == "female":
        return "Female"
    return raw.title() if raw else raw


def get_group_name(g: int) -> str:
    if g < len(gender_names):
        return _standardize_group_name(gender_names[g])
    return f"Group {g}"


def get_label_name(y: int) -> str:
    if y < len(selected_profession_names):
        return str(selected_profession_names[y])
    return str(y)


def get_group_color(g: int) -> str:
    palette = GROUP_COLOR_PALETTE if "GROUP_COLOR_PALETTE" in globals() else ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
    return palette[int(g) % len(palette)]


def style_axis(ax, grid_axis: str = "y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.22, linewidth=0.8)
    ax.set_axisbelow(True)


def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=260, bbox_inches="tight")
    plt.show()
    plt.close()


def _apply_temperature_global(probs: np.ndarray, T: float, eps: float = 1e-12) -> np.ndarray:
    # Apply temperature scaling to probabilities (row-wise) while staying in probability space.
    probs = np.asarray(probs, dtype=float)
    safe = np.clip(probs, eps, 1.0)
    if abs(T - 1.0) < 1e-12:
        return safe / safe.sum(axis=1, keepdims=True)
    powered = safe ** (1.0 / float(T))
    return powered / powered.sum(axis=1, keepdims=True)


def _score_method() -> str:
    return str(CONFORMAL_SCORE).strip().lower()


def _random_u(n: int) -> np.ndarray:
    # Randomization used by APS/RAPS/SAPS. If disabled, returns u=0.5 for determinism.
    if bool(SCORE_RANDOMIZE):
        rng = np.random.default_rng(int(SCORE_RANDOM_SEED))
        return rng.random(n)
    return np.full(n, 0.5, dtype=float)


def score_axis_label() -> str:
    m = _score_method()
    if m in ("simple", "1-p", "1p"):
        return r"True-label nonconformity score $s=1-\hat p_y(x)$"
    if m == "raps":
        return "True-label nonconformity score (RAPS)"
    if m == "saps":
        return "True-label nonconformity score (SAPS)"
    return "True-label nonconformity score"


def all_label_scores(probs: np.ndarray, method: str = None) -> np.ndarray:
    # Return an (n, K) matrix of nonconformity scores s(x_i, y).
    # Lower score => label is more 'conforming' for that x_i.
    # Prediction set at threshold q: { y : s(x, y) <= q }.
    method = (method or _score_method()).strip().lower()
    probs = _apply_temperature_global(probs, float(SCORE_TEMPERATURE))
    probs = np.asarray(probs, dtype=float)

    if method in ("simple", "1-p", "1p"):
        return 1.0 - probs

    n, K = probs.shape
    u = _random_u(n).reshape(-1, 1)

    # rank (1=largest prob) for each label y
    order = np.argsort(-probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(n)[:, None], order] = np.arange(1, K + 1)

    # cumulative mass ahead of each label (rho)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumsum = np.cumsum(sorted_probs, axis=1)
    rho_sorted = np.concatenate([np.zeros((n, 1)), cumsum[:, :-1]], axis=1)
    rho = rho_sorted[np.arange(n)[:, None], ranks - 1]

    if method == "raps":
        lam = float(RAPS_LAMBDA)
        k_reg = int(RAPS_K_REG)
        return rho + u * probs + lam * np.maximum(ranks - k_reg, 0)

    if method == "saps":
        lam = float(SAPS_LAMBDA)
        pmax = probs.max(axis=1, keepdims=True)
        base = pmax + lam * (ranks - 2 + u)
        top = u * probs
        return np.where(ranks == 1, top, base)

    raise ValueError(f"Unknown CONFORMAL_SCORE={method!r}. Use 'simple', 'raps', or 'saps'.")


def true_label_scores(probs: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores = all_label_scores(probs)
    return scores[np.arange(len(y)), y]


def split_conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(np.partition(scores, k - 1)[k - 1])


def pooled_threshold(scores: np.ndarray, alpha: float) -> float:
    return split_conformal_quantile(scores, alpha)


def group_thresholds(scores: np.ndarray, groups: np.ndarray, alpha: float):
    out = {}
    for g in sorted(np.unique(groups)):
        out[int(g)] = split_conformal_quantile(scores[groups == g], alpha)
    return out


def empirical_cdf_at_threshold(scores_group: np.ndarray, threshold: float) -> float:
    return float(np.mean(scores_group <= threshold))


def empirical_group_coverage(scores: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else threshold_by_group
        out[int(g)] = empirical_cdf_at_threshold(scores[mask], threshold)
    return out


def average_set_size_at_threshold(scores_all: np.ndarray, threshold: float) -> float:
    return float(np.mean((scores_all <= threshold).sum(axis=1)))


def average_group_set_size(scores_all: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else threshold_by_group
        out[int(g)] = average_set_size_at_threshold(scores_all[mask], threshold)
    return out


def group_weights(groups: np.ndarray):
    unique, counts = np.unique(groups, return_counts=True)
    weights = counts / counts.sum()
    return {int(g): float(w) for g, w in zip(unique, weights)}


def weighted_sd(values_by_group, weights_by_group):
    groups = sorted(values_by_group.keys())
    vals = np.array([values_by_group[g] for g in groups], dtype=float)
    w = np.array([weights_by_group[g] for g in groups], dtype=float)
    mean = np.sum(w * vals)
    var = np.sum(w * (vals - mean) ** 2)
    return float(np.sqrt(var))


def rms_by_group(values_by_group, weights_by_group):
    groups = sorted(values_by_group.keys())
    vals = np.array([values_by_group[g] for g in groups], dtype=float)
    w = np.array([weights_by_group[g] for g in groups], dtype=float)
    return float(np.sqrt(np.sum(w * vals ** 2)))


def size_curve_from_scores(scores_all_group: np.ndarray, grid: np.ndarray) -> np.ndarray:
    cutoffs = np.sort(scores_all_group.reshape(-1))
    n_examples = scores_all_group.shape[0]
    return np.searchsorted(cutoffs, grid, side="right") / max(n_examples, 1)


def attainable_size_support(scores_all_group: np.ndarray):
    flat_scores = np.sort(np.asarray(scores_all_group, dtype=float).reshape(-1))
    n_examples = int(np.asarray(scores_all_group).shape[0])
    if flat_scores.size == 0 or n_examples <= 0:
        return np.asarray([], dtype=float), np.asarray([], dtype=float)
    thresholds, counts = np.unique(flat_scores, return_counts=True)
    mean_set_sizes = np.cumsum(counts).astype(float) / float(n_examples)
    return thresholds.astype(float), mean_set_sizes.astype(float)


def solve_threshold_for_target_size(
    scores_all_group: np.ndarray,
    target_size: float,
    reference_threshold: float = None,
):
    thresholds, mean_set_sizes = attainable_size_support(scores_all_group)
    if thresholds.size == 0:
        return 0.0, 0.0
    target_size = float(target_size)
    diffs = np.abs(mean_set_sizes - target_size)
    best = np.flatnonzero(diffs == diffs.min())
    if reference_threshold is not None and best.size > 1:
        ref = float(reference_threshold)
        best = np.asarray([best[np.argmin(np.abs(thresholds[best] - ref))]])
    idx = int(best[0])
    return float(thresholds[idx]), float(mean_set_sizes[idx])


def weighted_second_moment(values_by_group, weights_by_group) -> float:
    groups = sorted(values_by_group.keys())
    return float(sum(float(weights_by_group[g]) * float(values_by_group[g]) ** 2 for g in groups))


def rms_weighted_coefficient(coeff_by_group, gap_by_group, weights_by_group) -> float:
    denom = weighted_second_moment(gap_by_group, weights_by_group)
    if denom <= 1e-18:
        return 0.0
    numer = float(sum(
        float(weights_by_group[g]) * (float(coeff_by_group[g]) ** 2) * (float(gap_by_group[g]) ** 2)
        for g in sorted(coeff_by_group.keys())
    ))
    return float(math.sqrt(max(numer, 0.0) / denom))


def segment_average_density_proxy(epsilon: float, threshold_gap: float) -> float:
    gap = float(threshold_gap)
    if abs(gap) <= 1e-18:
        return 0.0
    return float(abs(float(epsilon)) / abs(gap))


def barplot_by_group(values_by_group, title, ylabel, path, refline=None):
    groups = sorted(values_by_group.keys())
    labels = [get_group_name(g) for g in groups]
    values = [values_by_group[g] for g in groups]
    colors = [get_group_color(g) for g in groups]

    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    bars = ax.bar(labels, values, color=colors, width=0.62)
    if refline is not None:
        ax.axhline(refline, linestyle="--", linewidth=1.2, color="black")
    style_axis(ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    pad = max(0.002, 0.12 * max(1e-12, np.max(np.abs(values))))
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            value + (pad if value >= 0 else -pad),
            f"{value:+.4f}" if refline == 0.0 else f"{value:.4f}",
            ha="center",
            va="bottom" if value >= 0 else "top",
            fontsize=9,
        )
    savefig(path)


def apply_group_temperature(probs: np.ndarray, groups: np.ndarray, temperature_map: dict, eps: float = 1e-12) -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    groups = np.asarray(groups)
    adjusted = np.empty_like(probs)
    for g in np.unique(groups):
        mask = groups == g
        T = float(temperature_map[int(g)])
        safe = np.clip(probs[mask], eps, 1.0)
        if abs(T - 1.0) < 1e-12:
            adjusted[mask] = safe / safe.sum(axis=1, keepdims=True)
        else:
            powered = safe ** (1.0 / T)
            adjusted[mask] = powered / powered.sum(axis=1, keepdims=True)
    return adjusted


def finite_difference_size_slope(size_curve: np.ndarray, grid: np.ndarray, threshold: float) -> float:
    idx = int(np.argmin(np.abs(grid - threshold)))
    left = max(idx - 1, 0)
    right = min(idx + 1, len(grid) - 1)
    if right == left:
        return 0.0
    return float((size_curve[right] - size_curve[left]) / (grid[right] - grid[left]))


def ecdf_xy(values: np.ndarray):
    x = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(x) + 1, dtype=float) / len(x)
    return x, y


def centered_limits(values, center=0.0, frac=0.18, min_halfspan=0.003):
    arr = np.asarray(list(values), dtype=float)
    if arr.size == 0:
        return center - min_halfspan, center + min_halfspan
    max_dev = float(np.max(np.abs(arr - center)))
    half = max(min_halfspan, (1.0 + frac) * max_dev)
    return center - half, center + half


def local_window(values, pad_frac=0.25, min_pad=0.02, lo_floor=0.0, hi_cap=None):
    arr = np.asarray(list(values), dtype=float)
    if arr.size == 0:
        return float(lo_floor), (1.0 if hi_cap is None else float(hi_cap))
    lo = float(arr.min())
    hi = float(arr.max())
    span = hi - lo
    pad = max(min_pad, pad_frac * max(span, 1e-6))
    lo_out = max(float(lo_floor), lo - pad)
    hi_out = hi + pad if hi_cap is None else min(float(hi_cap), hi + pad)
    return lo_out, hi_out


def local_curve_limits(curves, grid, thresholds, margin_frac=0.18, min_margin=0.01):
    y_vals = []
    for curve in curves:
        for t in thresholds:
            y_vals.append(float(np.interp(t, grid, curve)))
    if not y_vals:
        return 0.0, 1.0
    lo = min(y_vals)
    hi = max(y_vals)
    span = hi - lo
    margin = max(min_margin, margin_frac * max(span, 1e-6))
    return max(0.0, lo - margin), hi + margin


def plot_ecdf_with_inset(scores_by_group, q, q_g, alpha, path):
    groups_sorted = sorted(scores_by_group.keys())
    fig, ax = plt.subplots(figsize=(7.0, 4.6))

    for g in groups_sorted:
        x, y = ecdf_xy(scores_by_group[g])
        ax.step(x, y, where="post", linewidth=2.0, color=get_group_color(g), label=get_group_name(g))
        ax.axvline(q_g[g], linestyle="--", linewidth=1.3, color=get_group_color(g), alpha=0.9)
    ax.axvline(q, linestyle="-", linewidth=1.6, color="black")
    ax.axhline(1 - alpha, linestyle=":", linewidth=1.2, color="black", alpha=0.75)

    style_axis(ax)
    ax.set_xlabel(score_axis_label())
    ax.set_ylabel(r"Empirical CDF $\hat F_g(s)$")
    ax.set_title("A1. Calibration score ECDFs with pooled and group thresholds")
    max_score = max([float(np.max(scores_by_group[g])) for g in groups_sorted] + [float(q)] + [float(v) for v in q_g.values()])
    ax.set_xlim(0.0, max(1e-6, 1.05 * max_score))
    ax.set_ylim(0.0, 1.01)

    inset = inset_axes(ax, width="43%", height="48%", loc="lower right", borderpad=1.0)
    for g in groups_sorted:
        x, y = ecdf_xy(scores_by_group[g])
        inset.step(x, y, where="post", linewidth=1.7, color=get_group_color(g))
        inset.axvline(q_g[g], linestyle="--", linewidth=1.1, color=get_group_color(g), alpha=0.9)
    inset.axvline(q, linestyle="-", linewidth=1.3, color="black")
    inset.axhline(1 - alpha, linestyle=":", linewidth=1.0, color="black", alpha=0.75)
    x_lo, x_hi = local_window(list(q_g.values()) + [q], pad_frac=0.8, min_pad=0.012)
    inset.set_xlim(x_lo, x_hi)
    y_lo, y_hi = centered_limits([1 - alpha], center=1 - alpha, frac=0.0, min_halfspan=0.06)
    inset.set_ylim(max(0.0, y_lo), min(1.0, y_hi))
    style_axis(inset)
    inset.set_title("Zoom near thresholds", fontsize=9)
    inset.tick_params(labelsize=8)

    legend_handles = [
        Line2D([0], [0], color=get_group_color(g), linewidth=2.0, label=get_group_name(g))
        for g in groups_sorted
    ] + [
        Line2D([0], [0], color="black", linewidth=1.6, label="Pooled threshold"),
        Line2D([0], [0], color="black", linestyle=":", linewidth=1.2, label="Target coverage"),
    ]
    ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=2, frameon=False)
    savefig(path)


def plot_group_coverage_bars(Fg_q, alpha, path):
    groups_sorted = sorted(Fg_q.keys())
    labels = [get_group_name(g) for g in groups_sorted]
    values = [Fg_q[g] for g in groups_sorted]
    colors = [get_group_color(g) for g in groups_sorted]
    target = 1 - alpha

    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    bars = ax.bar(labels, values, color=colors, width=0.62)
    ax.axhline(target, linestyle="--", linewidth=1.3, color="black", label=f"Target = {target:.2f}")
    style_axis(ax)
    ax.set_ylabel("Empirical coverage")
    ax.set_title("A2. Pooled-policy group coverage")
    y_lo, y_hi = centered_limits(values + [target], center=target, frac=0.25, min_halfspan=0.01)
    ax.set_ylim(max(0.0, y_lo), min(1.0, y_hi))
    for bar, value in zip(bars, values):
        delta = value - target
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            value + 0.0015,
            f"{value:.4f}\n({delta:+.4f})",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.24))
    savefig(path)


def plot_local_setsize_curves(size_curves, grid, q, q_g, alpha, path, title, tau_g=None, lambda_target=None):
    groups_sorted = sorted(size_curves.keys())
    fig, ax = plt.subplots(figsize=(6.8, 4.4))

    threshold_pool = [q] + [q_g[g] for g in groups_sorted]
    if tau_g is not None:
        threshold_pool += [tau_g[g] for g in groups_sorted]
    x_lo, x_hi = local_window(threshold_pool, pad_frac=0.75, min_pad=0.015)
    y_lo, y_hi = local_curve_limits(
        [size_curves[g] for g in groups_sorted],
        grid,
        thresholds=threshold_pool,
        margin_frac=0.22,
        min_margin=0.006,
    )
    if lambda_target is not None:
        y_lo = min(y_lo, lambda_target - 0.01)
        y_hi = max(y_hi, lambda_target + 0.01)

    for g in groups_sorted:
        ax.plot(grid, size_curves[g], linewidth=2.0, color=get_group_color(g), label=get_group_name(g))
        ax.axvline(q_g[g], linestyle="--", linewidth=1.2, color=get_group_color(g), alpha=0.9)
        if tau_g is not None:
            ax.axvline(tau_g[g], linestyle=":", linewidth=1.2, color=get_group_color(g), alpha=0.95)
    ax.axvline(q, linestyle="-", linewidth=1.5, color="black", label="Pooled threshold")
    if lambda_target is not None:
        ax.axhline(lambda_target, linestyle="-.", linewidth=1.5, color="black", label="Common size target")

    style_axis(ax)
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(max(0.0, y_lo), y_hi)
    ax.set_xlabel(r"Threshold $t$")
    ax.set_ylabel(r"Expected set size $\ell_g(t)$")
    ax.set_title(title)

    legend_handles = [
        Line2D([0], [0], color=get_group_color(g), linewidth=2.0, label=get_group_name(g))
        for g in groups_sorted
    ] + [
        Line2D([0], [0], color="black", linewidth=1.5, label="Pooled threshold"),
        Line2D([0], [0], color="black", linestyle="--", linewidth=1.2, label="Group threshold"),
    ]
    if tau_g is not None:
        legend_handles.append(Line2D([0], [0], color="black", linestyle=":", linewidth=1.2, label="Equalized-size threshold"))
    if lambda_target is not None:
        legend_handles.append(Line2D([0], [0], color="black", linestyle="-.", linewidth=1.5, label="Common size target"))
    ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=2, frameon=False)
    savefig(path)


def plot_policy_transition(Fg_q, l_pooled, Fg_qg, lambda_g, alpha, path):
    groups_sorted = sorted(Fg_q.keys())
    fig, ax = plt.subplots(figsize=(6.8, 4.7))

    x_vals = []
    y_vals = []
    for g in groups_sorted:
        color = get_group_color(g)
        x0, y0 = Fg_q[g], l_pooled[g]
        x1, y1 = Fg_qg[g], lambda_g[g]
        x_vals += [x0, x1]
        y_vals += [y0, y1]
        ax.scatter(x0, y0, s=75, color=color, marker="o", zorder=3)
        ax.scatter(x1, y1, s=85, color=color, marker="X", zorder=3)
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=1.4, shrinkA=5, shrinkB=5),
        )
        ax.text(x1 + 0.0006, y1 + 0.0015, get_group_name(g), color=color, fontsize=9)
    ax.axvline(1 - alpha, linestyle="--", linewidth=1.2, color="black")
    style_axis(ax)
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Expected set size")
    ax.set_title("B3. Pooled to group-wise policy transition")
    x_lo, x_hi = centered_limits(x_vals + [1 - alpha], center=1 - alpha, frac=0.35, min_halfspan=0.01)
    ax.set_xlim(max(0.0, x_lo), min(1.0, x_hi))
    y_margin = max(0.005, 0.18 * max(1e-6, max(y_vals) - min(y_vals)))
    ax.set_ylim(max(0.0, min(y_vals) - y_margin), max(y_vals) + y_margin)
    legend_handles = [
        Line2D([0], [0], marker="o", color="black", linestyle="None", markersize=7, label="Pooled"),
        Line2D([0], [0], marker="X", color="black", linestyle="None", markersize=8, label="Group-wise"),
        Line2D([0], [0], color="black", linestyle="--", linewidth=1.2, label="Target coverage"),
    ]
    ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.26), ncol=3, frameon=False)
    savefig(path)


def plot_size_gap_and_cov_change(lambda_g, lambda_target, delta_cov_g, path):
    groups_sorted = sorted(lambda_g.keys())
    labels = [get_group_name(g) for g in groups_sorted]
    colors = [get_group_color(g) for g in groups_sorted]
    size_gap = [lambda_g[g] - lambda_target for g in groups_sorted]
    cov_change = [delta_cov_g[g] for g in groups_sorted]

    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.8))
    for ax, values, title, ylabel in [
        (axes[0], size_gap, "Size gap before equalization", r"$\lambda_g - \lambda$"),
        (axes[1], cov_change, "Coverage change after equalization", r"$F_g(\tau_g)-F_g(q_g)$"),
    ]:
        bars = ax.bar(labels, values, color=colors, width=0.62)
        ax.axhline(0.0, linestyle="--", linewidth=1.1, color="black")
        style_axis(ax)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel)
        y_lo, y_hi = centered_limits(values, center=0.0, frac=0.25, min_halfspan=0.004)
        ax.set_ylim(y_lo, y_hi)
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                value + (0.0008 if value >= 0 else -0.0008),
                f"{value:+.4f}",
                ha="center",
                va="bottom" if value >= 0 else "top",
                fontsize=9,
            )

    fig.suptitle("C3. Size gap and induced coverage change", y=1.03, fontsize=12)
    plt.tight_layout()
    plt.savefig(path, dpi=260, bbox_inches="tight")
    plt.show()
    plt.close()


def _make_score_grid(max_score: float, step: float, max_points: int = 5001) -> np.ndarray:
    max_score = float(max_score)
    step = float(step)
    if max_score <= 0:
        return np.array([0.0, 1.0], dtype=float)
    n_points = int(max_score / max(step, 1e-9)) + 1
    if n_points > max_points:
        return np.linspace(0.0, max_score, num=max_points, dtype=float)
    return np.arange(0.0, max_score + step / 2.0, step, dtype=float)

def run_single_alpha_experiment(
    alpha: float,
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    g_cal: np.ndarray,
    probs_test: np.ndarray,
    y_test: np.ndarray,
    g_test: np.ndarray,
    plot_dir: Path = None,
    grid_step: float = 0.001,
    make_plots: bool = True,
    tag_prefix: str = "",
):
    alpha_tag = f"{tag_prefix}alpha_{str(alpha).replace('.', 'p')}" if tag_prefix else f"alpha_{str(alpha).replace('.', 'p')}"
    alpha_plot_dir = None
    if make_plots and plot_dir is not None:
        alpha_plot_dir = plot_dir / alpha_tag
        alpha_plot_dir.mkdir(parents=True, exist_ok=True)

    scores_cal = true_label_scores(probs_cal, y_cal)
    scores_test = true_label_scores(probs_test, y_test)
    scores_all_test = all_label_scores(probs_test)

    q = pooled_threshold(scores_cal, alpha)
    q_g = group_thresholds(scores_cal, g_cal, alpha)

    p_g = group_weights(g_test)
    groups_sorted = sorted(p_g.keys())

    # Experiment A
    Fg_q = empirical_group_coverage(scores_test, g_test, q)
    eps_g = {g: Fg_q[g] - (1 - alpha) for g in groups_sorted}
    sigma_delta = weighted_sd(q_g, p_g)
    rms_cov = rms_by_group(eps_g, p_g)

    # Experiment B
    l_pooled = average_group_set_size(scores_all_test, g_test, q)
    lambda_g = average_group_set_size(scores_all_test, g_test, q_g)
    delta_l_g = {g: lambda_g[g] - l_pooled[g] for g in groups_sorted}
    rms_size = rms_by_group(delta_l_g, p_g)

    max_score_grid = float(np.max(scores_all_test))
    grid = _make_score_grid(max_score=max_score_grid, step=grid_step)
    scores_all_test_by_group = {g: scores_all_test[g_test == g] for g in groups_sorted}
    size_curves = {g: size_curve_from_scores(scores_all_test_by_group[g], grid) for g in groups_sorted}
    size_slopes_at_q = {
        g: finite_difference_size_slope(size_curves[g], grid, q)
        for g in groups_sorted
    }
    size_slopes_at_qg = {
        g: finite_difference_size_slope(size_curves[g], grid, q_g[g])
        for g in groups_sorted
    }
    c_eff_proxy = float(np.sum([p_g[g] * abs(size_slopes_at_qg[g]) for g in groups_sorted]))

    q_gap_g = {g: q - q_g[g] for g in groups_sorted}
    segment_density_g = {
        g: segment_average_density_proxy(eps_g[g], q_gap_g[g])
        for g in groups_sorted
    }
    m_eff_segment_proxy = rms_weighted_coefficient(segment_density_g, q_gap_g, p_g)
    v_eff_proxy = rms_weighted_coefficient(
        {g: abs(size_slopes_at_qg[g]) for g in groups_sorted},
        q_gap_g,
        p_g,
    )

    Fg_qg = empirical_group_coverage(scores_test, g_test, q_g)

    # Experiment C
    lambda_target = float(sum(p_g[g] * lambda_g[g] for g in groups_sorted))
    tau_g = {}
    size_equalized = {}
    for g in groups_sorted:
        tau_g[g], size_equalized[g] = solve_threshold_for_target_size(
            scores_all_test_by_group[g],
            lambda_target,
            reference_threshold=q_g[g],
        )

    Fg_tau = empirical_group_coverage(scores_test, g_test, tau_g)
    delta_cov_g = {g: Fg_tau[g] - Fg_qg[g] for g in groups_sorted}
    sigma_lambda = weighted_sd(lambda_g, p_g)
    rms_cov_from_size = rms_by_group(delta_cov_g, p_g)

    cov_change_per_size = {}
    lambda_gap_g = {g: lambda_g[g] - lambda_target for g in groups_sorted}
    for g in groups_sorted:
        denom = lambda_gap_g[g]
        if abs(denom) < 1e-12:
            cov_change_per_size[g] = 0.0
        else:
            cov_change_per_size[g] = delta_cov_g[g] / denom
    kappa_eff_proxy = float(np.sum([p_g[g] * abs(cov_change_per_size[g]) for g in groups_sorted]))
    kappa_eff_rms_proxy = rms_weighted_coefficient(
        {g: abs(cov_change_per_size[g]) for g in groups_sorted},
        lambda_gap_g,
        p_g,
    )

    if make_plots and alpha_plot_dir is not None:
        scores_by_group = {g: scores_cal[g_cal == g] for g in groups_sorted}
        plot_ecdf_with_inset(
            scores_by_group=scores_by_group,
            q=q,
            q_g=q_g,
            alpha=alpha,
            path=alpha_plot_dir / "A1_grouped_score_distributions.png",
        )

        plot_group_coverage_bars(
            Fg_q=Fg_q,
            alpha=alpha,
            path=alpha_plot_dir / "A2_group_coverage_pooled.png",
        )

        barplot_by_group(
            eps_g,
            title="A3. Deviation from target coverage under pooled policy",
            ylabel=r"$\hat F_g(q) - (1-\alpha)$",
            path=alpha_plot_dir / "A3_group_miscoverage_pooled.png",
            refline=0.0,
        )

        plot_local_setsize_curves(
            size_curves=size_curves,
            grid=grid,
            q=q,
            q_g=q_g,
            alpha=alpha,
            path=alpha_plot_dir / "B1_groupwise_setsize_curves.png",
            title="B1. Local view of set-size curves near operating thresholds",
        )

        barplot_by_group(
            delta_l_g,
            title="B2. Set-size change after group-wise calibration",
            ylabel=r"$\ell_g(q_g)-\ell_g(q)$",
            path=alpha_plot_dir / "B2_size_change_barplot.png",
            refline=0.0,
        )

        plot_policy_transition(
            Fg_q=Fg_q,
            l_pooled=l_pooled,
            Fg_qg=Fg_qg,
            lambda_g=lambda_g,
            alpha=alpha,
            path=alpha_plot_dir / "B3_coverage_size_policy_compare.png",
        )

        plot_local_setsize_curves(
            size_curves=size_curves,
            grid=grid,
            q=q,
            q_g=q_g,
            alpha=alpha,
            tau_g=tau_g,
            lambda_target=lambda_target,
            path=alpha_plot_dir / "C1_setsize_curves_common_target.png",
            title="C1. Equalized-size target in a local threshold window",
        )

        barplot_by_group(
            delta_cov_g,
            title="C2. Coverage change after equalized-size adjustment",
            ylabel=r"$\hat F_g(\tau_g)-\hat F_g(q_g)$",
            path=alpha_plot_dir / "C2_coverage_distortion_after_equalized_size.png",
            refline=0.0,
        )

        plot_size_gap_and_cov_change(
            lambda_g=lambda_g,
            lambda_target=lambda_target,
            delta_cov_g=delta_cov_g,
            path=alpha_plot_dir / "C3_size_gap_vs_coverage_change.png",
        )

    group_rows = []
    for g in groups_sorted:
        row = {
            "alpha": alpha,
            "group": g,
            "group_name": get_group_name(g),
            "p_g": p_g[g],
            "q_g": q_g[g],
            "coverage_pooled": Fg_q[g],
            "epsilon_pooled": eps_g[g],
            "size_pooled": l_pooled[g],
            "coverage_groupwise": Fg_qg[g],
            "lambda_g": lambda_g[g],
            "delta_size_from_groupwise": delta_l_g[g],
            "tau_g": tau_g[g],
            "size_equalized": size_equalized[g],
            "coverage_equalized_size": Fg_tau[g],
            "delta_cov_from_equalized_size": delta_cov_g[g],
            "segment_density_g": segment_density_g[g],
            "size_slope_at_q": size_slopes_at_q[g],
            "size_slope_at_qg": size_slopes_at_qg[g],
            "cov_change_per_size": cov_change_per_size[g],
        }
        group_rows.append(row)

    group_df = pd.DataFrame(group_rows)
    summary = {
        "alpha": alpha,
        "q_pooled": q,
        "sigma_delta": sigma_delta,
        "c_eff_proxy": c_eff_proxy,
        "c_eff_times_sigma_delta": c_eff_proxy * sigma_delta,
        "m_eff_segment_proxy": m_eff_segment_proxy,
        "m_eff_segment_times_sigma_delta": m_eff_segment_proxy * sigma_delta,
        "v_eff_proxy": v_eff_proxy,
        "v_eff_times_sigma_delta": v_eff_proxy * sigma_delta,
        "rms_cov_pooled": rms_cov,
        "lambda_target": lambda_target,
        "rms_size_from_groupwise": rms_size,
        "sigma_lambda": sigma_lambda,
        "kappa_eff_proxy": kappa_eff_proxy,
        "kappa_eff_times_sigma_lambda": kappa_eff_proxy * sigma_lambda,
        "kappa_eff_rms_proxy": kappa_eff_rms_proxy,
        "kappa_eff_rms_times_sigma_lambda": kappa_eff_rms_proxy * sigma_lambda,
        "rms_cov_from_equalized_size": rms_cov_from_size,
    }

    return {
        "summary": summary,
        "group_df": group_df,
        "scores_cal": scores_cal,
        "scores_test": scores_test,
    }


## Run one seed end-to-end

This function creates the seed-specific FACET split, extracts zero-shot CLIP probabilities if needed, exports the arrays required by the plan, and runs the complete alpha sweep.

For FACET, the seed affects the **class-stratified calibration / calibration-validation / test split**. The CLIP base model itself is fixed.

In [ ]:

def _mean_set_size_from_probs_and_q(probs: np.ndarray, q: float) -> float:
    scores_all = all_label_scores(probs)
    return average_set_size_at_threshold(scores_all, q)


def tune_global_score_temperature(
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    probs_val: np.ndarray,
    y_val: np.ndarray,
    alpha: float,
    candidate_temperatures=None,
    coverage_tol: float = None,
):
    candidate_temperatures = list(candidate_temperatures or AUTO_TUNE_TEMPERATURE_GRID)
    coverage_tol = float(AUTO_TUNE_COVERAGE_TOL if coverage_tol is None else coverage_tol)
    target = 1.0 - float(alpha)

    rows = []
    for T in candidate_temperatures:
        probs_cal_T = _apply_temperature_global(probs_cal, float(T))
        probs_val_T = _apply_temperature_global(probs_val, float(T))
        scores_cal_T = true_label_scores(probs_cal_T, y_cal)
        scores_val_T = true_label_scores(probs_val_T, y_val)
        q_T = pooled_threshold(scores_cal_T, alpha)
        cov_val = empirical_cdf_at_threshold(scores_val_T, q_T)
        set_size_val = _mean_set_size_from_probs_and_q(probs_val_T, q_T)
        rows.append({
            "temperature": float(T),
            "q_cal": float(q_T),
            "coverage_val": float(cov_val),
            "coverage_gap_val": float(cov_val - target),
            "abs_coverage_gap_val": float(abs(cov_val - target)),
            "mean_set_size_val": float(set_size_val),
            "feasible": bool(cov_val >= target - coverage_tol),
        })

    tuning_df = pd.DataFrame(rows).sort_values("temperature").reset_index(drop=True)
    feasible = tuning_df[tuning_df["feasible"]].copy()
    if len(feasible) > 0:
        feasible = feasible.sort_values(
            ["mean_set_size_val", "abs_coverage_gap_val", "temperature"],
            ascending=[True, True, True],
        ).reset_index(drop=True)
        chosen = feasible.iloc[0].to_dict()
    else:
        fallback = tuning_df.sort_values(
            ["abs_coverage_gap_val", "mean_set_size_val", "temperature"],
            ascending=[True, True, True],
        ).reset_index(drop=True)
        chosen = fallback.iloc[0].to_dict()

    return float(chosen["temperature"]), tuning_df


def run_seed(seed: int):
    print("=" * 80)
    print(f"Running seed {seed}")
    set_global_seed(seed)

    seed_dir = RUN_ROOT / f"seed_{seed}"
    arrays_dir = seed_dir / "arrays"
    plots_dir = seed_dir / "plots"
    tables_dir = seed_dir / "tables"
    temp_dir = seed_dir / "temperature_sweep"
    seed_dir.mkdir(parents=True, exist_ok=True)
    arrays_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(parents=True, exist_ok=True)

    splits = build_splits_for_seed(seed)
    split_summary = pd.concat(
        [summarize_split(ds, split_name) for split_name, ds in splits.items()],
        ignore_index=True,
    )
    split_summary.to_csv(tables_dir / "split_summary.csv", index=False)

    split_meta = {
        "run_name": RUN_NAME,
        "experiment_dataset": "FACET",
        "seed": seed,
        "dataset_name": DATASET_NAME,
        "clip_model_name": CLIP_MODEL_NAME,
        "metadata_path": METADATA_PATH,
        "experiment_mode": EXPERIMENT_MODE,
        "label_ids": selected_profession_ids,
        "label_names": selected_profession_names,
        "group_names": gender_names,
        "facet_group_variant": FACET_GROUP_VARIANT,
        "facet_split_stratify_mode": FACET_SPLIT_STRATIFY_MODE,
        "facet_age_margin_min": FACET_AGE_MARGIN_MIN,
        "facet_image_mode": FACET_IMAGE_MODE,
        "facet_bbox_expand_ratio": FACET_BBOX_EXPAND_RATIO,
        "facet_fixed_labels": FACET_FIXED_LABELS,
        "split_sizes": {name: len(ds) for name, ds in splits.items()},
        "facet_split_size_mode": FACET_SPLIT_SIZE_MODE,
        "facet_target_split_sizes_published": FACET_TARGET_SPLIT_SIZES_PUBLISHED,
        "limits": LIMITS,
        "primary_alpha": PRIMARY_ALPHA,
        "alphas": ALPHAS,
        "grid_step": GRID_STEP,
        "temperature_sweep_group": TEMPERATURE_SWEEP_GROUP,
        "temperature_sweep_group_name": gender_names[TEMPERATURE_SWEEP_GROUP] if TEMPERATURE_SWEEP_GROUP < len(gender_names) else str(TEMPERATURE_SWEEP_GROUP),
        "temperature_sweep_values": TEMPERATURE_SWEEP_VALUES,
        "conformal_score": CONFORMAL_SCORE,
        "clip_prompt_templates": CLIP_PROMPT_TEMPLATES,
        "auto_tune_score_temperature": AUTO_TUNE_SCORE_TEMPERATURE,
        "auto_tune_temperature_grid": AUTO_TUNE_TEMPERATURE_GRID,
    }
    with open(seed_dir / "split_meta.json", "w") as f:
        json.dump(to_jsonable(split_meta), f, indent=2)

    probs_calval_path = arrays_dir / "probs_calibration_validation.npy"
    y_calval_path = arrays_dir / "y_calibration_validation.npy"
    g_calval_path = arrays_dir / "g_calibration_validation.npy"

    probs_cal_path = arrays_dir / "probs_cal.npy"
    probs_test_path = arrays_dir / "probs_test.npy"
    y_cal_path = arrays_dir / "y_cal.npy"
    y_test_path = arrays_dir / "y_test.npy"
    g_cal_path = arrays_dir / "g_cal.npy"
    g_test_path = arrays_dir / "g_test.npy"

    if RUN_MODE == "train_and_analyze":
        inference_summary, arrays = extract_probabilities_for_seed(seed=seed, splits=splits)

        with open(seed_dir / "inference_summary.json", "w") as f:
            json.dump(to_jsonable(inference_summary), f, indent=2)

        np.save(probs_calval_path, arrays["probs_calibration_validation"])
        np.save(y_calval_path, arrays["y_calibration_validation"])
        np.save(g_calval_path, arrays["g_calibration_validation"])

        np.save(probs_cal_path, arrays["probs_cal"])
        np.save(probs_test_path, arrays["probs_test"])
        np.save(y_cal_path, arrays["y_cal"])
        np.save(y_test_path, arrays["y_test"])
        np.save(g_cal_path, arrays["g_cal"])
        np.save(g_test_path, arrays["g_test"])

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elif RUN_MODE == "analyze_only":
        required_paths = [
            probs_cal_path, probs_test_path, y_cal_path, y_test_path, g_cal_path, g_test_path
        ]
        missing = [str(p) for p in required_paths if not p.exists()]
        if missing:
            raise FileNotFoundError(
                "RUN_MODE='analyze_only' was selected, but these files are missing:\n" + "\n".join(missing)
            )
    else:
        raise ValueError("RUN_MODE must be 'train_and_analyze' or 'analyze_only'.")

    probs_calval = np.load(probs_calval_path)
    y_calval = np.load(y_calval_path)
    g_calval = np.load(g_calval_path)

    probs_cal = np.load(probs_cal_path)
    probs_test = np.load(probs_test_path)
    y_cal = np.load(y_cal_path)
    y_test = np.load(y_test_path)
    g_cal = np.load(g_cal_path)
    g_test = np.load(g_test_path)


    chosen_score_temperature = float(SCORE_TEMPERATURE)
    tuning_df = pd.DataFrame()
    if bool(AUTO_TUNE_SCORE_TEMPERATURE):
        if abs(float(SCORE_TEMPERATURE) - 1.0) > 1e-12:
            raise ValueError(
                "AUTO_TUNE_SCORE_TEMPERATURE=True expects SCORE_TEMPERATURE=1.0. "
                "Set SCORE_TEMPERATURE back to 1.0 and let the tuning routine choose T."
            )
        chosen_score_temperature, tuning_df = tune_global_score_temperature(
            probs_cal=probs_cal,
            y_cal=y_cal,
            probs_val=probs_calval,
            y_val=y_calval,
            alpha=PRIMARY_ALPHA,
            candidate_temperatures=AUTO_TUNE_TEMPERATURE_GRID,
            coverage_tol=AUTO_TUNE_COVERAGE_TOL,
        )
        probs_calval = _apply_temperature_global(probs_calval, chosen_score_temperature)
        probs_cal = _apply_temperature_global(probs_cal, chosen_score_temperature)
        probs_test = _apply_temperature_global(probs_test, chosen_score_temperature)
        tuning_df.to_csv(tables_dir / "score_temperature_tuning.csv", index=False)
        with open(seed_dir / "chosen_score_temperature.json", "w") as f:
            json.dump(
                {
                    "chosen_score_temperature": float(chosen_score_temperature),
                    "primary_alpha": float(PRIMARY_ALPHA),
                    "grid": list(map(float, AUTO_TUNE_TEMPERATURE_GRID)),
                    "coverage_tol": float(AUTO_TUNE_COVERAGE_TOL),
                },
                f,
                indent=2,
            )
        print(f"[seed {seed}] chosen global score temperature = {chosen_score_temperature:.3f}")
    else:
        print(f"[seed {seed}] using fixed SCORE_TEMPERATURE = {float(SCORE_TEMPERATURE):.3f}")

    cal_scores = true_label_scores(probs_cal, y_cal)
    test_scores = true_label_scores(probs_test, y_test)
    np.save(arrays_dir / "cal_scores.npy", cal_scores)
    np.save(arrays_dir / "test_scores.npy", test_scores)

    all_group_tables = []
    all_summaries = []
    for alpha in ALPHAS:
        make_plots = (alpha == PRIMARY_ALPHA)
        result = run_single_alpha_experiment(
            alpha=alpha,
            probs_cal=probs_cal,
            y_cal=y_cal,
            g_cal=g_cal,
            probs_test=probs_test,
            y_test=y_test,
            g_test=g_test,
            plot_dir=plots_dir,
            grid_step=GRID_STEP,
            make_plots=make_plots,
        )
        group_df = result["group_df"].copy()
        group_df.insert(0, "seed", seed)
        all_group_tables.append(group_df)

        summary = dict(result["summary"])
        summary["seed"] = seed
        summary["chosen_score_temperature"] = float(chosen_score_temperature)
        summary["is_primary_alpha"] = bool(abs(alpha - PRIMARY_ALPHA) < 1e-12)
        all_summaries.append(summary)

    group_metrics_df = pd.concat(all_group_tables, ignore_index=True)
    summary_df = pd.DataFrame(all_summaries)
    interpretation_df = summary_df.copy()
    interpretation_df["A_visible"] = interpretation_df["rms_cov_pooled"] > 0
    if {"m_eff_segment_times_sigma_delta", "rms_cov_pooled"}.issubset(interpretation_df.columns):
        interpretation_df["A_minus_proxy"] = (
            interpretation_df["rms_cov_pooled"] - interpretation_df["m_eff_segment_times_sigma_delta"]
        )
    interpretation_df["B_visible"] = interpretation_df["rms_size_from_groupwise"] > 0
    interpretation_df["C_visible"] = interpretation_df["rms_cov_from_equalized_size"] > 0

    group_metrics_df.to_csv(tables_dir / "group_metrics_by_alpha.csv", index=False)
    summary_df.to_csv(tables_dir / "alpha_summary.csv", index=False)
    interpretation_df.to_csv(tables_dir / "interpretation_table.csv", index=False)

    primary_summary_df = summary_df[np.isclose(summary_df["alpha"], PRIMARY_ALPHA)].copy()
    primary_group_df = group_metrics_df[np.isclose(group_metrics_df["alpha"], PRIMARY_ALPHA)].copy()
    primary_summary_df.to_csv(tables_dir / "primary_alpha_summary.csv", index=False)
    primary_group_df.to_csv(tables_dir / "primary_alpha_group_metrics.csv", index=False)

    temperature_summary_df = pd.DataFrame()
    if RUN_TEMPERATURE_SWEEP:
        temp_rows = []
        for temp_value in TEMPERATURE_SWEEP_VALUES:
            temperature_map = {int(g): 1.0 for g in sorted(np.unique(g_cal))}
            temperature_map[int(TEMPERATURE_SWEEP_GROUP)] = float(temp_value)

            probs_cal_temp = apply_group_temperature(probs_cal, g_cal, temperature_map, eps=TEMPERATURE_EPS)
            probs_test_temp = apply_group_temperature(probs_test, g_test, temperature_map, eps=TEMPERATURE_EPS)

            temp_result = run_single_alpha_experiment(
                alpha=PRIMARY_ALPHA,
                probs_cal=probs_cal_temp,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test_temp,
                y_test=y_test,
                g_test=g_test,
                plot_dir=None,
                grid_step=GRID_STEP,
                make_plots=False,
            )
            row = dict(temp_result["summary"])
            row["seed"] = seed
            row["temperature_group"] = int(TEMPERATURE_SWEEP_GROUP)
            row["temperature_group_name"] = (
                gender_names[int(TEMPERATURE_SWEEP_GROUP)]
                if int(TEMPERATURE_SWEEP_GROUP) < len(gender_names) else str(TEMPERATURE_SWEEP_GROUP)
            )
            row["temperature_value"] = float(temp_value)
            temp_rows.append(row)

        temperature_summary_df = pd.DataFrame(temp_rows)
        temperature_summary_df.to_csv(tables_dir / "temperature_sweep_summary.csv", index=False)

    return {
        "seed": seed,
        "seed_dir": str(seed_dir),
        "summary_df": summary_df,
        "group_metrics_df": group_metrics_df,
        "primary_summary_df": primary_summary_df,
        "primary_group_df": primary_group_df,
        "temperature_summary_df": temperature_summary_df,
        "score_temperature_tuning_df": tuning_df,
        "chosen_score_temperature": float(chosen_score_temperature),
    }

## Execute all requested seeds

Start with `SEEDS = [0]`. After the pipeline works, increase the list for exploratory or main-result runs.


In [ ]:
all_seed_results = []
for seed in SEEDS:
    result = run_seed(seed)
    all_seed_results.append(result)

all_summary_df = pd.concat([item["summary_df"] for item in all_seed_results], ignore_index=True)
all_group_metrics_df = pd.concat([item["group_metrics_df"] for item in all_seed_results], ignore_index=True)
all_primary_summary_df = pd.concat([item["primary_summary_df"] for item in all_seed_results], ignore_index=True)
all_primary_group_df = pd.concat([item["primary_group_df"] for item in all_seed_results], ignore_index=True)

temperature_frames = [item["temperature_summary_df"] for item in all_seed_results if len(item["temperature_summary_df"]) > 0]
all_temperature_summary_df = (
    pd.concat(temperature_frames, ignore_index=True)
    if len(temperature_frames) > 0 else
    pd.DataFrame()
)

tuning_frames = [
    item["score_temperature_tuning_df"].assign(seed=item["seed"])
    for item in all_seed_results
    if len(item.get("score_temperature_tuning_df", pd.DataFrame())) > 0
]
all_score_temperature_tuning_df = (
    pd.concat(tuning_frames, ignore_index=True)
    if len(tuning_frames) > 0 else
    pd.DataFrame()
)

def _agg_numeric(df, group_cols, value_cols):
    if df.empty:
        return pd.DataFrame()
    group_cols = [c for c in list(group_cols or []) if c in df.columns]
    use_cols = [c for c in value_cols if c in df.columns]
    if len(use_cols) == 0:
        return pd.DataFrame()

    if len(group_cols) == 0:
        grouped = df[use_cols].agg(["mean", "std", "count"]).T.reset_index()
        grouped = grouped.rename(columns={"index": "metric"})
        return grouped

    grouped = df.groupby(group_cols, dropna=False)[use_cols].agg(["mean", "std", "count"]).reset_index()
    grouped.columns = [
        "_".join([str(x) for x in col if str(x) not in ["", "nan"]]).rstrip("_")
        if isinstance(col, tuple) else str(col)
        for col in grouped.columns
    ]
    return grouped

primary_summary_agg = _agg_numeric(
    all_primary_summary_df,
    group_cols=["alpha"] if "alpha" in all_primary_summary_df.columns else [],
    value_cols=[
        "q_pooled",
        "sigma_delta",
        "c_eff_proxy",
        "c_eff_times_sigma_delta",
        "m_eff_segment_proxy",
        "m_eff_segment_times_sigma_delta",
        "v_eff_proxy",
        "v_eff_times_sigma_delta",
        "rms_cov_pooled",
        "lambda_target",
        "rms_size_from_groupwise",
        "sigma_lambda",
        "kappa_eff_proxy",
        "kappa_eff_times_sigma_lambda",
        "kappa_eff_rms_proxy",
        "kappa_eff_rms_times_sigma_lambda",
        "rms_cov_from_equalized_size",
        "chosen_score_temperature",
    ],
)

primary_group_agg = _agg_numeric(
    all_primary_group_df,
    group_cols=["group", "group_name"] if {"group", "group_name"}.issubset(all_primary_group_df.columns) else [],
    value_cols=[
        "p_g",
        "q_g",
        "coverage_pooled",
        "epsilon_pooled",
        "size_pooled",
        "coverage_groupwise",
        "lambda_g",
        "delta_size_from_groupwise",
        "tau_g",
        "size_equalized",
        "coverage_equalized_size",
        "delta_cov_from_equalized_size",
        "size_slope_at_qg",
        "cov_change_per_size",
    ],
)

if not all_temperature_summary_df.empty:
    temperature_group_cols = []
    if "temperature_group" in all_temperature_summary_df.columns:
        temperature_group_cols.append("temperature_group")
    if "temperature_group_name" in all_temperature_summary_df.columns:
        temperature_group_cols.append("temperature_group_name")
    if "temperature_value" in all_temperature_summary_df.columns:
        temperature_group_cols.append("temperature_value")

    temperature_agg = _agg_numeric(
        all_temperature_summary_df,
        group_cols=temperature_group_cols,
        value_cols=[
            "sigma_delta",
            "c_eff_proxy",
            "c_eff_times_sigma_delta",
            "m_eff_segment_proxy",
            "m_eff_segment_times_sigma_delta",
            "v_eff_proxy",
            "v_eff_times_sigma_delta",
            "rms_cov_pooled",
            "lambda_target",
            "rms_size_from_groupwise",
            "sigma_lambda",
            "kappa_eff_proxy",
            "kappa_eff_times_sigma_lambda",
            "kappa_eff_rms_proxy",
            "kappa_eff_rms_times_sigma_lambda",
            "rms_cov_from_equalized_size",
        ],
    )
else:
    temperature_agg = pd.DataFrame()

all_summary_df.to_csv(RUN_ROOT / "all_seeds_alpha_summary.csv", index=False)
all_group_metrics_df.to_csv(RUN_ROOT / "all_seeds_group_metrics.csv", index=False)
all_primary_summary_df.to_csv(RUN_ROOT / "all_seeds_primary_alpha_summary.csv", index=False)
all_primary_group_df.to_csv(RUN_ROOT / "all_seeds_primary_alpha_group_metrics.csv", index=False)
if not all_temperature_summary_df.empty:
    all_temperature_summary_df.to_csv(RUN_ROOT / "all_seeds_temperature_sweep_summary.csv", index=False)
if not primary_summary_agg.empty:
    primary_summary_agg.to_csv(RUN_ROOT / "aggregate_primary_alpha_summary.csv", index=False)
if not primary_group_agg.empty:
    primary_group_agg.to_csv(RUN_ROOT / "aggregate_primary_alpha_group_metrics.csv", index=False)
if not temperature_agg.empty:
    temperature_agg.to_csv(RUN_ROOT / "aggregate_temperature_sweep_summary.csv", index=False)
if not all_score_temperature_tuning_df.empty:
    all_score_temperature_tuning_df.to_csv(RUN_ROOT / "all_seeds_score_temperature_tuning.csv", index=False)

print("Saved combined tables to:")
print(RUN_ROOT / "all_seeds_alpha_summary.csv")
print(RUN_ROOT / "all_seeds_group_metrics.csv")
print(RUN_ROOT / "all_seeds_primary_alpha_summary.csv")
print(RUN_ROOT / "all_seeds_primary_alpha_group_metrics.csv")
if not all_temperature_summary_df.empty:
    print(RUN_ROOT / "all_seeds_temperature_sweep_summary.csv")
if not primary_summary_agg.empty:
    print(RUN_ROOT / "aggregate_primary_alpha_summary.csv")
if not primary_group_agg.empty:
    print(RUN_ROOT / "aggregate_primary_alpha_group_metrics.csv")
if not temperature_agg.empty:
    print(RUN_ROOT / "aggregate_temperature_sweep_summary.csv")
if not all_score_temperature_tuning_df.empty:
    print(RUN_ROOT / "all_seeds_score_temperature_tuning.csv")

display(all_primary_summary_df)
display(all_primary_group_df.head(20))
if not all_temperature_summary_df.empty:
    display(all_temperature_summary_df.head(20))

print("\nAggregated primary summary across seeds:")
display(primary_summary_agg if not primary_summary_agg.empty else pd.DataFrame())

print("\nAggregated primary group metrics across seeds:")
display(primary_group_agg if not primary_group_agg.empty else pd.DataFrame())

if not all_score_temperature_tuning_df.empty:
    print("\nScore-temperature tuning traces:")
    display(all_score_temperature_tuning_df.head(20))

## Main fixed-alpha paper plots, alpha robustness, and controlled temperature sweep

In [ ]:
# --- Robust helpers added for stand-alone plotting in the last cell ---
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
except Exception:
    inset_axes = None
    mark_inset = None

if "TEMPERATURE_EPS" not in globals():
    TEMPERATURE_EPS = 1e-12
if "SCORE_RANDOMIZE" not in globals():
    SCORE_RANDOMIZE = False
if "SCORE_RANDOM_SEED" not in globals():
    SCORE_RANDOM_SEED = 0
if "CONFORMAL_SCORE" not in globals():
    CONFORMAL_SCORE = "simple"

def apply_group_temperature(probs, groups, temperature_map, eps=TEMPERATURE_EPS):
    probs = np.asarray(probs, dtype=float)
    groups = np.asarray(groups)
    out = np.clip(probs, eps, 1.0)
    for g, T in dict(temperature_map).items():
        mask = groups == g
        if not np.any(mask):
            continue
        temp = max(float(T), eps)
        powered = np.power(np.clip(out[mask], eps, 1.0), 1.0 / temp)
        denom = powered.sum(axis=1, keepdims=True)
        denom = np.clip(denom, eps, None)
        out[mask] = powered / denom
    return out

def _temperature_xlabel(temp_group):
    if "gender_names" in globals() and isinstance(gender_names, dict):
        gname = gender_names.get(temp_group, f"group {temp_group}")
        return rf"temperature $T$ for {gname}"
    return rf"temperature $T$ for group {temp_group}"

def _latex_metric(metric_name):
    mapping = {
        "rms_cov_pooled": r"$\mathrm{RMS}\,\epsilon_g(q)$",
        "rms_size_from_groupwise": r"$\mathrm{RMS}\,\Delta \ell_g$",
        "rms_cov_from_equalized_size": r"$\mathrm{RMS}\,\Delta F_g$",
        "sigma_delta": r"$\sigma_\Delta$",
        "sigma_lambda": r"$\sigma_\lambda$",
        "c_eff_times_sigma_delta": r"$c_{\mathrm{eff}}\,\sigma_\Delta$",
        "m_eff_segment_times_sigma_delta": r"$m_{\mathrm{eff}}^{\mathrm{seg}}\,\sigma_\Delta$",
        "kappa_eff_times_sigma_lambda": r"$\kappa_{\mathrm{eff}}\,\sigma_\lambda$",
        "kappa_eff_rms_times_sigma_lambda": r"$\kappa_{\mathrm{eff}}^{\mathrm{rms}}\,\sigma_\lambda$",
        "temperature_value": r"$T$",
        "alpha": r"$\alpha$",
    }
    return mapping.get(metric_name, str(metric_name).replace("_", " "))

def _annot_label(name, value):
    try:
        v = float(value)
    except Exception:
        return str(value)
    if name == "alpha":
        return rf"$\alpha={v:.3g}$"
    if name == "temperature_value":
        return rf"$T={v:.2f}$"
    return f"{v:.3g}"
# --- End robust helpers ---


# ============================================================
# Main fixed-alpha paper plots, alpha robustness,
# and controlled temperature sweep
# Paper-facing Panel A now uses the segment-based
# proxy m_eff_segment_times_sigma_delta rather than
# c_eff_times_sigma_delta.
# Stand-alone local Jupyter version for FACET:
# this cell no longer depends on an external plot.py file.
# ============================================================

from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# ------------------------------------------------------------
# Figure-generation controls
# ------------------------------------------------------------
GENERATE_BASELINE_ALPHA_SWEEP = True
GENERATE_TEMPERATURE_ALPHA_GRID = True   # every temperature × every alpha
GENERATE_TEMPERATURE_PRIMARY_ONLY = False  # optional extra convenience folder

BASELINE_6PANEL_DIRNAME = "paper_6panel_by_alpha"
TEMP_GRID_6PANEL_DIRNAME = "paper_6panel_temp_alpha_grid"
TEMP_PRIMARY_DIRNAME = "paper_6panel_by_temperature_primary_alpha"

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 350,
    "font.size": 8.5,
    "axes.titlesize": 9,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.2,
    "mathtext.fontset": "dejavusans",
    "lines.linewidth": 1.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

# ------------------------------------------------------------
# Stand-alone plot.py-style helpers
# ------------------------------------------------------------
PALETTE = GROUP_COLOR_PALETTE if "GROUP_COLOR_PALETTE" in globals() else ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]

def _p6_style_axis(ax, grid_axis: str = "y") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.18, linewidth=0.7)
    ax.set_axisbelow(True)

def _p6_standardize_group_name(name: str) -> str:
    raw = str(name).strip()
    lower = raw.lower()
    if lower == "male":
        return "Male"
    if lower == "female":
        return "Female"
    return raw.title() if raw else raw

def _p6_get_group_name(g: int, gender_names_local) -> str:
    if 0 <= int(g) < len(gender_names_local):
        return _p6_standardize_group_name(gender_names_local[int(g)])
    return f"Group {g}"

def _p6_get_group_color(g: int) -> str:
    return PALETTE[int(g) % len(PALETTE)]

def _p6_apply_temperature_global(probs: np.ndarray, T: float, eps: float = 1e-12) -> np.ndarray:
    probs = np.asarray(probs, dtype=float)
    safe = np.clip(probs, eps, 1.0)
    if abs(T - 1.0) < 1e-12:
        return safe / safe.sum(axis=1, keepdims=True)
    powered = safe ** (1.0 / float(T))
    return powered / powered.sum(axis=1, keepdims=True)


def _p6_score_method() -> str:
    return str(CONFORMAL_SCORE).strip().lower() if "CONFORMAL_SCORE" in globals() else "simple"


def _p6_random_u(n: int) -> np.ndarray:
    if ("SCORE_RANDOMIZE" in globals()) and bool(SCORE_RANDOMIZE):
        rng = np.random.default_rng(int(globals().get("SCORE_RANDOM_SEED", 0)))
        return rng.random(n)
    return np.full(n, 0.5, dtype=float)


def _p6_all_label_scores(probs: np.ndarray, method: str = None) -> np.ndarray:
    method = (method or _p6_score_method()).strip().lower()
    T = float(globals().get("SCORE_TEMPERATURE", 1.0))
    probs = _p6_apply_temperature_global(probs, T)
    probs = np.asarray(probs, dtype=float)

    if method in ("simple", "1-p", "1p"):
        return 1.0 - probs

    n, K = probs.shape
    u = _p6_random_u(n).reshape(-1, 1)

    order = np.argsort(-probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(n)[:, None], order] = np.arange(1, K + 1)

    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumsum = np.cumsum(sorted_probs, axis=1)
    rho_sorted = np.concatenate([np.zeros((n, 1)), cumsum[:, :-1]], axis=1)
    rho = rho_sorted[np.arange(n)[:, None], ranks - 1]

    if method == "raps":
        lam = float(globals().get("RAPS_LAMBDA", 0.2))
        k_reg = int(globals().get("RAPS_K_REG", 5))
        return rho + u * probs + lam * np.maximum(ranks - k_reg, 0)

    if method == "saps":
        lam = float(globals().get("SAPS_LAMBDA", 0.2))
        pmax = probs.max(axis=1, keepdims=True)
        base = pmax + lam * (ranks - 2 + u)
        top = u * probs
        return np.where(ranks == 1, top, base)

    raise ValueError(f"Unknown CONFORMAL_SCORE={method!r}. Use 'simple', 'raps', or 'saps'.")


def _p6_true_label_scores(probs: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores = _p6_all_label_scores(probs)
    return scores[np.arange(len(y)), y]

def _p6_split_conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(np.partition(scores, k - 1)[k - 1])

def _p6_group_thresholds(scores: np.ndarray, groups: np.ndarray, alpha: float):
    out = {}
    for g in sorted(np.unique(groups)):
        out[int(g)] = _p6_split_conformal_quantile(scores[groups == g], alpha)
    return out

def _p6_empirical_cdf_at_threshold(scores_group: np.ndarray, threshold: float) -> float:
    return float(np.mean(scores_group <= threshold))

def _p6_empirical_group_coverage(scores: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else float(threshold_by_group)
        out[int(g)] = _p6_empirical_cdf_at_threshold(scores[mask], threshold)
    return out

def _p6_average_set_size_at_threshold(scores_all: np.ndarray, threshold: float) -> float:
    return float(np.mean((scores_all <= threshold).sum(axis=1)))

def _p6_average_group_set_size(scores_all: np.ndarray, groups: np.ndarray, threshold_by_group):
    out = {}
    for g in sorted(np.unique(groups)):
        mask = groups == g
        threshold = threshold_by_group[g] if isinstance(threshold_by_group, dict) else threshold_by_group
        out[int(g)] = _p6_average_set_size_at_threshold(scores_all[mask], threshold)
    return out

def _p6_group_weights(groups: np.ndarray):
    unique, counts = np.unique(groups, return_counts=True)
    counts = counts.astype(float)
    counts /= counts.sum()
    return {int(g): float(w) for g, w in zip(unique, counts)}

def _p6_weighted_mean(values_by_group, weights_by_group) -> float:
    return float(sum(weights_by_group[g] * values_by_group[g] for g in values_by_group))

def _p6_size_curve_from_scores(scores_all_group: np.ndarray, grid: np.ndarray) -> np.ndarray:
    cutoffs = np.sort(scores_all_group.reshape(-1))
    n_examples = scores_all_group.shape[0]
    return np.searchsorted(cutoffs, grid, side="right") / max(n_examples, 1)

def _p6_empirical_ecdf(scores: np.ndarray):
    xs = np.sort(scores)
    ys = np.arange(1, len(xs) + 1, dtype=float) / len(xs)
    return xs, ys

def _p6_find_tau_from_size_curve(grid: np.ndarray, size_curve: np.ndarray, lambda_target: float) -> float:
    idx = int(np.argmin(np.abs(size_curve - lambda_target)))
    return float(grid[idx])

def _p6_compute_primary_metrics(
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    g_cal: np.ndarray,
    probs_test: np.ndarray,
    y_test: np.ndarray,
    g_test: np.ndarray,
    alpha: float,
    grid_step: float = 0.001,
):
    cal_scores = _p6_true_label_scores(probs_cal, y_cal)
    test_scores = _p6_true_label_scores(probs_test, y_test)
    test_scores_all = _p6_all_label_scores(probs_test)

    q_pooled = _p6_split_conformal_quantile(cal_scores, alpha)
    q_g = _p6_group_thresholds(cal_scores, g_cal, alpha)

    coverage_pooled = _p6_empirical_group_coverage(test_scores, g_test, q_pooled)
    coverage_groupwise = _p6_empirical_group_coverage(test_scores, g_test, q_g)

    size_pooled = _p6_average_group_set_size(probs_test, g_test, q_pooled)
    lambda_g = _p6_average_group_set_size(probs_test, g_test, q_g)
    delta_size = {g: lambda_g[g] - size_pooled[g] for g in lambda_g}

    weights = _p6_group_weights(g_test)
    lambda_target = _p6_weighted_mean(lambda_g, weights)

    grid = np.arange(0.0, 1.0 + grid_step, grid_step)
    size_curves = {}
    tau_g = {}
    size_equalized = {}
    coverage_equalized = {}
    delta_cov = {}

    for g in sorted(np.unique(g_test)):
        g = int(g)
        probs_group = probs_test[g_test == g]
        size_curve = _p6_size_curve_from_scores(probs_group, grid)
        size_curves[g] = size_curve
        tau = _p6_find_tau_from_size_curve(grid, size_curve, lambda_target)
        tau_g[g] = tau
        size_equalized[g] = _p6_average_set_size_at_threshold(probs_group, tau)
        coverage_equalized[g] = _p6_empirical_cdf_at_threshold(test_scores[g_test == g], tau)
        delta_cov[g] = coverage_equalized[g] - coverage_groupwise[g]

    return {
        "alpha": alpha,
        "q_pooled": q_pooled,
        "q_g": q_g,
        "coverage_pooled": coverage_pooled,
        "coverage_groupwise": coverage_groupwise,
        "size_pooled": size_pooled,
        "lambda_g": lambda_g,
        "delta_size": delta_size,
        "lambda_target": lambda_target,
        "tau_g": tau_g,
        "size_equalized": size_equalized,
        "coverage_equalized": coverage_equalized,
        "delta_cov": delta_cov,
        "weights": weights,
        "cal_scores": cal_scores,
        "test_scores": test_scores,
        "grid": grid,
        "size_curves": size_curves,
    }

def _p6_ecdf_plot_with_inset(ax, metrics, g_cal: np.ndarray, gender_names_local, target_coverage: float) -> None:
    cal_scores = metrics["cal_scores"]
    q_pooled = metrics["q_pooled"]
    q_g = metrics["q_g"]

    groups = sorted(np.unique(g_cal))
    for g in groups:
        g = int(g)
        xs, ys = _p6_empirical_ecdf(cal_scores[g_cal == g])
        ax.step(xs, ys, where="post", color=_p6_get_group_color(g), label=_p6_get_group_name(g, gender_names_local))
        ax.axvline(q_g[g], color=_p6_get_group_color(g), linestyle="--", linewidth=1.1)

    ax.axvline(q_pooled, color="black", linewidth=1.5, label=r"$q$ (pooled)")
    _p6_style_axis(ax, grid_axis="both")
    ax.set_xlabel(r"True-label score $S(X,Y)$")
    ax.set_ylabel(r"ECDF $F_g(t)$")
    ax.set_title(r"A1. Group ECDFs $F_g$ on calibration scores")
    ax.legend(frameon=False, loc="lower right")

    all_qs = [q_pooled] + [q_g[int(g)] for g in groups]
    x0 = max(0.0, min(all_qs) - 0.03)
    x1 = min(1.0, max(all_qs) + 0.03)
    y0 = max(0.0, target_coverage - 0.05)
    y1 = min(1.0, target_coverage + 0.05)

    axins = inset_axes(ax, width="48%", height="52%", loc="lower center", borderpad=1.1)
    for g in groups:
        g = int(g)
        xs, ys = _p6_empirical_ecdf(cal_scores[g_cal == g])
        axins.step(xs, ys, where="post", color=_p6_get_group_color(g))
        axins.axvline(q_g[g], color=_p6_get_group_color(g), linestyle="--", linewidth=1.0)
    axins.axvline(q_pooled, color="black", linewidth=1.3)
    axins.set_xlim(x0, x1)
    axins.set_ylim(y0, y1)
    axins.set_xticks([])
    axins.set_yticks([])
    axins.grid(True, axis="both", alpha=0.15, linewidth=0.5)
    for spine in axins.spines.values():
        spine.set_linewidth(0.8)
    mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.6", lw=0.6)

def _p6_centered_barplot(ax, values_by_group, gender_names_local, ylabel: str, title: str) -> None:
    groups = sorted(values_by_group)
    labels = [_p6_get_group_name(g, gender_names_local) for g in groups]
    vals = [values_by_group[g] for g in groups]
    colors = [_p6_get_group_color(g) for g in groups]
    bars = ax.bar(labels, vals, color=colors, width=0.62)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    _p6_style_axis(ax, grid_axis="y")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    max_abs = max(max(abs(v) for v in vals), 1e-6)
    pad = 0.10 * max_abs
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            v + (pad if v >= 0 else -pad),
            f"{v:+.4f}",
            ha="center",
            va="bottom" if v >= 0 else "top",
        )
    lim = max_abs * 1.32
    ax.set_ylim(-lim, lim)

def _p6_local_size_curve_plot(ax, metrics, g_test: np.ndarray, gender_names_local, title: str, mode: str = "groupwise") -> None:
    q_pooled = metrics["q_pooled"]
    q_g = metrics["q_g"]
    grid = metrics["grid"]
    size_curves = metrics["size_curves"]
    groups = sorted(np.unique(g_test))

    if mode == "groupwise":
        refs = [q_pooled] + [q_g[int(g)] for g in groups]
        y_refs = [metrics["size_pooled"][int(g)] for g in groups] + [metrics["lambda_g"][int(g)] for g in groups]
    else:
        refs = [metrics["tau_g"][int(g)] for g in groups] + [q_pooled]
        y_refs = [metrics["size_equalized"][int(g)] for g in groups] + [metrics["lambda_target"]]

    x0 = max(0.0, min(refs) - 0.03)
    x1 = min(1.0, max(refs) + 0.03)
    y0 = max(0.0, min(y_refs) - 0.015)
    y1 = max(y0 + 0.02, max(y_refs) + 0.015)

    for g in groups:
        g = int(g)
        ax.plot(grid, size_curves[g], color=_p6_get_group_color(g), label=_p6_get_group_name(g, gender_names_local))
    ax.axvline(q_pooled, color="black", linewidth=1.5, label=r"$q$ (pooled)")

    if mode == "groupwise":
        for g in groups:
            g = int(g)
            ax.axvline(q_g[g], color=_p6_get_group_color(g), linestyle="--", linewidth=1.1)
    else:
        for g in groups:
            g = int(g)
            ax.axvline(metrics["tau_g"][g], color=_p6_get_group_color(g), linestyle="--", linewidth=1.1)
        ax.axhline(metrics["lambda_target"], color="black", linewidth=1.2, linestyle="-", label=r"common target $\lambda$")

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    _p6_style_axis(ax, grid_axis="both")
    ax.set_xlabel("Threshold $t$")
    ax.set_ylabel(r"Expected set size $\ell_g(t)$")
    ax.set_title(title)
    ax.legend(frameon=False, loc="best")

# ------------------------------------------------------------
# Notebook-specific helpers
# ------------------------------------------------------------
def _alpha_tag(alpha: float) -> str:
    return f"alpha_{alpha:.2f}".replace(".", "p")

def _temp_tag(temp_value: float) -> str:
    return f"temp_{temp_value:.2f}".replace(".", "p")

def _safe_group_name(g: int) -> str:
    try:
        return _p6_standardize_group_name(gender_names[int(g)])
    except Exception:
        return f"Group {int(g)}"

def _load_seed_arrays(seed_dir: Path):
    arrays_dir = seed_dir / "arrays"
    probs_cal = np.load(arrays_dir / "probs_cal.npy")
    probs_test = np.load(arrays_dir / "probs_test.npy")
    y_cal = np.load(arrays_dir / "y_cal.npy")
    y_test = np.load(arrays_dir / "y_test.npy")
    g_cal = np.load(arrays_dir / "g_cal.npy")
    g_test = np.load(arrays_dir / "g_test.npy")
    return probs_cal, y_cal, g_cal, probs_test, y_test, g_test

def _save_figure(fig: plt.Figure, outpath: Path) -> None:
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(pad=0.45)
    fig.savefig(outpath, bbox_inches="tight")
    plt.close(fig)

def _six_panel_suptitle(seed: int, alpha: float, temp_value=None, temp_group=None) -> str:
    base = rf"FACET: mechanism plots ($\alpha={alpha:.2f}$)"
    if temp_value is None:
        return base
    return base + rf", temperature on {temp_group}: $T={float(temp_value):.2f}$"

def _combined_main_figure_neurips(
    *,
    metrics,
    g_cal: np.ndarray,
    g_test: np.ndarray,
    gender_names_local,
    outpath: Path,
    seed: int,
    alpha: float,
    temp_value=None,
    temp_group=None,
):
    target = 1.0 - alpha
    fig, axes = plt.subplots(2, 3, figsize=(10.1, 5.9))
    axes = axes.ravel()

    _p6_ecdf_plot_with_inset(axes[0], metrics, g_cal, gender_names_local, target)
    axes[0].set_title(r"A1. Score ECDFs $F_g$ on calibration split")

    _p6_centered_barplot(
        axes[1],
        {g: metrics["coverage_pooled"][g] - target for g in metrics["coverage_pooled"]},
        gender_names_local,
        ylabel=r"$F_g(q) - (1-\alpha)$",
        title=r"A2. Group miscoverage $\epsilon_g(q)$ under pooled threshold",
    )

    _p6_local_size_curve_plot(
        axes[2],
        metrics,
        g_test,
        gender_names_local,
        title=r"B1. Local set-size curves $\ell_g(t)$ near $q$ and $q_g$",
        mode="groupwise",
    )

    _p6_centered_barplot(
        axes[3],
        metrics["delta_size"],
        gender_names_local,
        ylabel=r"$\ell_g(q_g) - \ell_g(q)$",
        title=r"B2. Set-size shift $\Delta \ell_g = \ell_g(q_g)-\ell_g(q)$",
    )

    _p6_local_size_curve_plot(
        axes[4],
        metrics,
        g_test,
        gender_names_local,
        title=r"C1. Local curves $\ell_g(t)$ near $\tau_g$ and $\lambda$",
        mode="equalized",
    )

    _p6_centered_barplot(
        axes[5],
        metrics["delta_cov"],
        gender_names_local,
        ylabel=r"$F_g(\tau_g) - F_g(q_g)$",
        title=r"C2. Coverage shift $\Delta F_g = F_g(\tau_g)-F_g(q_g)$",
    )

    fig.suptitle(
        _six_panel_suptitle(seed=seed, alpha=alpha, temp_value=temp_value, temp_group=temp_group),
        y=1.01,
        fontsize=10,
    )
    _save_figure(fig, outpath)

def _generate_baseline_alpha_six_panels(
    *,
    seed: int,
    seed_dir: Path,
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    g_cal: np.ndarray,
    probs_test: np.ndarray,
    y_test: np.ndarray,
    g_test: np.ndarray,
):
    manifest_rows = []
    out_root = seed_dir / BASELINE_6PANEL_DIRNAME
    out_root.mkdir(parents=True, exist_ok=True)

    for alpha in ALPHAS:
        metrics = _p6_compute_primary_metrics(
            probs_cal=probs_cal,
            y_cal=y_cal,
            g_cal=g_cal,
            probs_test=probs_test,
            y_test=y_test,
            g_test=g_test,
            alpha=float(alpha),
            grid_step=float(GRID_STEP),
        )
        outdir = out_root / _alpha_tag(float(alpha))
        outpath = outdir / "main_ABC_6panel.png"
        _combined_main_figure_neurips(
            metrics=metrics,
            g_cal=g_cal,
            g_test=g_test,
            gender_names_local=gender_names,
            outpath=outpath,
            seed=seed,
            alpha=float(alpha),
            temp_value=None,
            temp_group=None,
        )
        manifest_rows.append({
            "seed": seed,
            "scenario": "baseline",
            "temperature_value": np.nan,
            "alpha": float(alpha),
            "output_png": str(outpath),
        })

    return manifest_rows

def _generate_temperature_alpha_grid_six_panels(
    *,
    seed: int,
    seed_dir: Path,
    probs_cal: np.ndarray,
    y_cal: np.ndarray,
    g_cal: np.ndarray,
    probs_test: np.ndarray,
    y_test: np.ndarray,
    g_test: np.ndarray,
):
    manifest_rows = []

    group_ids = sorted(np.unique(g_cal))
    temp_group = int(TEMPERATURE_SWEEP_GROUP)
    temp_group_name = _safe_group_name(temp_group)

    if GENERATE_TEMPERATURE_ALPHA_GRID:
        out_root = seed_dir / TEMP_GRID_6PANEL_DIRNAME
        out_root.mkdir(parents=True, exist_ok=True)

        for temp_value in TEMPERATURE_SWEEP_VALUES:
            temperature_map = {int(g): 1.0 for g in group_ids}
            temperature_map[temp_group] = float(temp_value)

            probs_cal_temp = apply_group_temperature(probs_cal, g_cal, temperature_map, eps=TEMPERATURE_EPS)
            probs_test_temp = apply_group_temperature(probs_test, g_test, temperature_map, eps=TEMPERATURE_EPS)

            for alpha in ALPHAS:
                metrics = _p6_compute_primary_metrics(
                    probs_cal=probs_cal_temp,
                    y_cal=y_cal,
                    g_cal=g_cal,
                    probs_test=probs_test_temp,
                    y_test=y_test,
                    g_test=g_test,
                    alpha=float(alpha),
                    grid_step=float(GRID_STEP),
                )
                outdir = out_root / _temp_tag(float(temp_value)) / _alpha_tag(float(alpha))
                outpath = outdir / "main_ABC_6panel.png"

                _combined_main_figure_neurips(
                    metrics=metrics,
                    g_cal=g_cal,
                    g_test=g_test,
                    gender_names_local=gender_names,
                    outpath=outpath,
                    seed=seed,
                    alpha=float(alpha),
                    temp_value=float(temp_value),
                    temp_group=temp_group_name,
                )
                manifest_rows.append({
                    "seed": seed,
                    "scenario": "temperature_alpha_grid",
                    "temperature_group": temp_group,
                    "temperature_group_name": temp_group_name,
                    "temperature_value": float(temp_value),
                    "alpha": float(alpha),
                    "output_png": str(outpath),
                })

    if GENERATE_TEMPERATURE_PRIMARY_ONLY:
        out_root = seed_dir / TEMP_PRIMARY_DIRNAME
        out_root.mkdir(parents=True, exist_ok=True)

        for temp_value in TEMPERATURE_SWEEP_VALUES:
            temperature_map = {int(g): 1.0 for g in group_ids}
            temperature_map[temp_group] = float(temp_value)

            probs_cal_temp = apply_group_temperature(probs_cal, g_cal, temperature_map, eps=TEMPERATURE_EPS)
            probs_test_temp = apply_group_temperature(probs_test, g_test, temperature_map, eps=TEMPERATURE_EPS)

            metrics = _p6_compute_primary_metrics(
                probs_cal=probs_cal_temp,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test_temp,
                y_test=y_test,
                g_test=g_test,
                alpha=float(PRIMARY_ALPHA),
                grid_step=float(GRID_STEP),
            )
            outdir = out_root / _temp_tag(float(temp_value))
            outpath = outdir / "main_ABC_6panel.png"

            _combined_main_figure_neurips(
                metrics=metrics,
                g_cal=g_cal,
                g_test=g_test,
                gender_names_local=gender_names,
                outpath=outpath,
                seed=seed,
                alpha=float(PRIMARY_ALPHA),
                temp_value=float(temp_value),
                temp_group=temp_group_name,
            )
            manifest_rows.append({
                "seed": seed,
                "scenario": "temperature_primary_alpha_only",
                "temperature_group": temp_group,
                "temperature_group_name": temp_group_name,
                "temperature_value": float(temp_value),
                "alpha": float(PRIMARY_ALPHA),
                "output_png": str(outpath),
            })

    return manifest_rows

# ------------------------------------------------------------
# Generate 6-panel figures seed by seed
# ------------------------------------------------------------
if len(all_seed_results) == 0:
    raise RuntimeError("all_seed_results is empty. Run the seed pipeline first.")

manifest = []

for item in all_seed_results:
    seed = int(item["seed"])
    seed_dir = Path(item["seed_dir"])
    print("=" * 100)
    print(f"[seed {seed}] generating stand-alone 6-panel figures")
    probs_cal, y_cal, g_cal, probs_test, y_test, g_test = _load_seed_arrays(seed_dir)

    if GENERATE_BASELINE_ALPHA_SWEEP:
        manifest.extend(
            _generate_baseline_alpha_six_panels(
                seed=seed,
                seed_dir=seed_dir,
                probs_cal=probs_cal,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test,
                y_test=y_test,
                g_test=g_test,
            )
        )

    if RUN_TEMPERATURE_SWEEP and (GENERATE_TEMPERATURE_ALPHA_GRID or GENERATE_TEMPERATURE_PRIMARY_ONLY):
        manifest.extend(
            _generate_temperature_alpha_grid_six_panels(
                seed=seed,
                seed_dir=seed_dir,
                probs_cal=probs_cal,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test,
                y_test=y_test,
                g_test=g_test,
            )
        )

manifest_df = pd.DataFrame(manifest)
manifest_path = RUN_ROOT / "main_ABC_6panel_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("\nSaved manifest:")
print(manifest_path)
display(manifest_df.head(20) if not manifest_df.empty else pd.DataFrame())

print("\nKey output folders:")
for item in all_seed_results:
    seed = int(item["seed"])
    seed_dir = Path(item["seed_dir"])
    if GENERATE_BASELINE_ALPHA_SWEEP:
        print(seed_dir / BASELINE_6PANEL_DIRNAME)
    if RUN_TEMPERATURE_SWEEP and GENERATE_TEMPERATURE_ALPHA_GRID:
        print(seed_dir / TEMP_GRID_6PANEL_DIRNAME)
    if RUN_TEMPERATURE_SWEEP and GENERATE_TEMPERATURE_PRIMARY_ONLY:
        print(seed_dir / TEMP_PRIMARY_DIRNAME)


# ------------------------------------------------------------
# Keep the existing summary / appendix plots as well
# ------------------------------------------------------------
primary_plot_dir = RUN_ROOT / "paper_primary_alpha"
robustness_plot_dir = RUN_ROOT / "appendix_alpha_robustness"
temp_plot_dir = RUN_ROOT / "appendix_temperature_sweep"

primary_plot_dir.mkdir(parents=True, exist_ok=True)
robustness_plot_dir.mkdir(parents=True, exist_ok=True)
if not all_temperature_summary_df.empty:
    temp_plot_dir.mkdir(parents=True, exist_ok=True)

def _safe_std(x):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return 0.0
    return float(x.std(ddof=1))

def _safe_mean(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(x.mean())

def _savefig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close()

def _mean_std_by(df, group_col, value_col):
    if df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=[group_col, "mean", "std"])
    out = (
        df.groupby(group_col, as_index=False)[value_col]
        .agg(mean=lambda s: _safe_mean(s), std=lambda s: _safe_std(s))
        .sort_values(group_col)
        .reset_index(drop=True)
    )
    return out

def _resolve_group_name(x):
    if isinstance(x, str):
        if x.strip() in {"0", "1"}:
            idx = int(x)
        else:
            return x
    else:
        try:
            idx = int(x)
        except Exception:
            return str(x)
    try:
        if "gender_names" in globals() and gender_names is not None and len(gender_names) > idx:
            return str(gender_names[idx])
    except Exception:
        pass
    fallback = {0: "Male", 1: "Female"}
    return fallback.get(idx, str(x))

def _normalize_group_df(group_df: pd.DataFrame) -> pd.DataFrame:
    gdf = group_df.copy()
    if "group_name" not in gdf.columns:
        if "group" in gdf.columns:
            gdf["group_name"] = gdf["group"].map(_resolve_group_name)
        else:
            gdf["group_name"] = np.arange(len(gdf)).astype(str)
    else:
        gdf["group_name"] = gdf["group_name"].map(_resolve_group_name)
    return gdf


def _plot_line_with_band(df, x_col, y_col, path, title, xlabel, ylabel, annotate=False):
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        print(f"[skip] Cannot plot {y_col}: missing data.")
        return

    stats = _mean_std_by(df, x_col, y_col)
    if stats.empty:
        print(f"[skip] Empty stats for {y_col}.")
        return

    stats = stats.sort_values(x_col).reset_index(drop=True)
    labels = [_annot_label(x_col, float(v)) if pd.notna(v) else str(v) for v in stats[x_col]]
    y = stats["mean"].to_numpy(dtype=float)
    s = stats["std"].fillna(0.0).to_numpy(dtype=float)
    xpos = np.arange(len(stats), dtype=float)

    fig, ax = plt.subplots(figsize=(3.45, 2.6))
    ax.bar(
        xpos,
        y,
        yerr=s if np.any(s > 0) else None,
        capsize=4 if np.any(s > 0) else 0,
        width=0.72,
        alpha=0.88,
    )
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if annotate:
        for xi, yi in zip(xpos, y):
            ax.annotate(f"{yi:.3f}", (xi, yi), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
    _savefig(path)

def _plot_metric_vs_metric(df, x_col, y_col, path, title, xlabel, ylabel, annotate_temp=False):
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        print(f"[skip] Cannot plot {x_col} vs {y_col}: missing data.")
        return

    if "temperature_value" not in df.columns:
        print(f"[skip] Cannot convert {x_col} vs {y_col} to bar plot without temperature_value.")
        return

    mean_df = (
        df.groupby("temperature_value", as_index=False)[[x_col, y_col]]
        .mean()
        .sort_values("temperature_value")
    )
    if mean_df.empty:
        print(f"[skip] Empty stats for {x_col} vs {y_col}.")
        return

    labels = [_annot_label("temperature_value", float(v)) for v in mean_df["temperature_value"]]
    y = mean_df[y_col].to_numpy(dtype=float)
    x_metric = mean_df[x_col].to_numpy(dtype=float)
    xpos = np.arange(len(mean_df), dtype=float)

    fig, ax = plt.subplots(figsize=(3.45, 2.6))
    ax.bar(xpos, y, width=0.72, alpha=0.88)
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.set_xlabel(_temperature_xlabel(TEMPERATURE_SWEEP_GROUP))
    ax.set_ylabel(ylabel)
    if annotate_temp:
        for xi, yi, xv in zip(xpos, y, x_metric):
            ax.annotate(
                f"{_latex_metric(x_col)}={xv:.3f}",
                (xi, yi),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                fontsize=8,
            )
    _savefig(path)


summary_alpha_cols = [
    c for c in [
        "sigma_delta",
        "m_eff_segment_times_sigma_delta",
        "rms_cov_pooled",
        "rms_size_from_groupwise",
        "sigma_lambda",
        "kappa_eff_times_sigma_lambda",
        "rms_cov_from_equalized_size",
    ] if c in all_summary_df.columns
]

summary_by_alpha = (
    all_summary_df
    .groupby("alpha", as_index=False)[summary_alpha_cols]
    .mean()
    .sort_values("alpha")
    if (not all_summary_df.empty and len(summary_alpha_cols) > 0)
    else pd.DataFrame()
)

group_summary_cols = [
    c for c in [
        "coverage_pooled",
        "coverage_groupwise",
        "coverage_equalized_size",
        "size_pooled",
        "lambda_g",
        "size_equalized",
        "epsilon_pooled",
        "delta_size_from_groupwise",
        "delta_cov_from_equalized_size",
    ] if c in all_primary_group_df.columns
]

primary_group_summary = (
    _normalize_group_df(all_primary_group_df)
    .groupby("group_name", as_index=False)[group_summary_cols]
    .mean()
    if (not all_primary_group_df.empty and len(group_summary_cols) > 0)
    else pd.DataFrame()
)

summary_by_alpha.to_csv(RUN_ROOT / "summary_by_alpha_mean_over_seeds.csv", index=False)
primary_group_summary.to_csv(RUN_ROOT / "summary_primary_group_mean_over_seeds.csv", index=False)

if not all_temperature_summary_df.empty:
    temp_cols = [
        c for c in [
            "sigma_delta",
            "m_eff_segment_times_sigma_delta",
            "rms_cov_pooled",
            "rms_size_from_groupwise",
            "sigma_lambda",
            "kappa_eff_times_sigma_lambda",
            "rms_cov_from_equalized_size",
        ] if c in all_temperature_summary_df.columns
    ]
    temperature_summary_mean = (
        all_temperature_summary_df
        .groupby("temperature_value", as_index=False)[temp_cols]
        .mean()
        .sort_values("temperature_value")
        if len(temp_cols) > 0 else pd.DataFrame()
    )
    temperature_summary_mean.to_csv(RUN_ROOT / "summary_temperature_sweep_mean_over_seeds.csv", index=False)
else:
    temperature_summary_mean = pd.DataFrame()

display(summary_by_alpha)
display(primary_group_summary)
if not temperature_summary_mean.empty:
    display(temperature_summary_mean)

if not all_primary_summary_df.empty:
    primary_summary_means = {}
    for c in ["rms_cov_pooled", "rms_size_from_groupwise", "rms_cov_from_equalized_size"]:
        if c in all_primary_summary_df.columns:
            primary_summary_means[c] = _safe_mean(all_primary_summary_df[c])

    if len(primary_summary_means) > 0:
        fig, ax = plt.subplots(figsize=(3.35, 2.6))
        cats = list(primary_summary_means.keys())
        vals = list(primary_summary_means.values())
        pretty = {
            "rms_cov_pooled": r"A: $\mathrm{RMS}\,\epsilon_g(q)$",
            "rms_size_from_groupwise": r"B: $\mathrm{RMS}\,\Delta \ell_g$",
            "rms_cov_from_equalized_size": r"C: $\mathrm{RMS}\,\Delta F_g$",
        }
        ax.bar([pretty.get(c, c) for c in cats], vals)
        ax.set_ylabel("Mean over seeds")
        ax.set_title(rf"Primary $\alpha={PRIMARY_ALPHA:.2f}$: A/B/C summaries")
        ax.tick_params(axis="x", rotation=10)
        _savefig(primary_plot_dir / "primary_alpha_ABC_summary_bars.png")

    proxy_means = {}
    for c in ["sigma_delta", "m_eff_segment_times_sigma_delta", "sigma_lambda", "kappa_eff_times_sigma_lambda"]:
        if c in all_primary_summary_df.columns:
            proxy_means[c] = _safe_mean(all_primary_summary_df[c])

    if len(proxy_means) > 0:
        fig, ax = plt.subplots(figsize=(4.15, 2.6))
        cats = list(proxy_means.keys())
        vals = list(proxy_means.values())
        ax.bar([_latex_metric(c) for c in cats], vals)
        ax.set_ylabel("Mean over seeds")
        ax.set_title(rf"Primary $\alpha={PRIMARY_ALPHA:.2f}$: heterogeneity and theorem-facing proxies")
        ax.tick_params(axis="x", rotation=20)
        _savefig(primary_plot_dir / "primary_alpha_proxy_summary_bars.png")

if not all_primary_group_df.empty:
    gdf = _normalize_group_df(all_primary_group_df).copy()

    coverage_cols = [c for c in ["coverage_pooled", "coverage_groupwise", "coverage_equalized_size"] if c in gdf.columns]
    if len(coverage_cols) > 0:
        fig, ax = plt.subplots(figsize=(4.2, 2.8))
        cats = list(gdf["group_name"].drop_duplicates())
        x = np.arange(len(cats))
        width = 0.8 / len(coverage_cols)
        for j, c in enumerate(coverage_cols):
            means = [_safe_mean(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            stds = [_safe_std(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            offset = (j - (len(coverage_cols) - 1) / 2) * width
            ax.bar(x + offset, means, width=width, yerr=stds, capsize=4, label=_latex_metric(c))
        ax.set_xticks(x)
        ax.set_xticklabels(cats)
        ax.set_ylabel("Coverage")
        ax.set_title(rf"Primary $\alpha={PRIMARY_ALPHA:.2f}$: coverage by group and policy")
        ax.legend(frameon=False)
        _savefig(primary_plot_dir / "primary_alpha_group_coverage_by_policy.png")

    size_cols = [c for c in ["size_pooled", "lambda_g", "size_equalized"] if c in gdf.columns]
    if len(size_cols) > 0:
        fig, ax = plt.subplots(figsize=(4.2, 2.8))
        cats = list(gdf["group_name"].drop_duplicates())
        x = np.arange(len(cats))
        width = 0.8 / len(size_cols)
        for j, c in enumerate(size_cols):
            means = [_safe_mean(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            stds = [_safe_std(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            offset = (j - (len(size_cols) - 1) / 2) * width
            ax.bar(x + offset, means, width=width, yerr=stds, capsize=4, label=_latex_metric(c))
        ax.set_xticks(x)
        ax.set_xticklabels(cats)
        ax.set_ylabel(r"Expected set size $\ell_g(t)$")
        ax.set_title(rf"Primary $\alpha={PRIMARY_ALPHA:.2f}$: set size by group and policy")
        ax.legend(frameon=False)
        _savefig(primary_plot_dir / "primary_alpha_group_size_by_policy.png")

    distortion_cols = [c for c in ["epsilon_pooled", "delta_size_from_groupwise", "delta_cov_from_equalized_size"] if c in gdf.columns]
    if len(distortion_cols) > 0:
        fig, ax = plt.subplots(figsize=(4.45, 2.8))
        cats = list(gdf["group_name"].drop_duplicates())
        x = np.arange(len(cats))
        width = 0.8 / len(distortion_cols)
        for j, c in enumerate(distortion_cols):
            means = [_safe_mean(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            stds = [_safe_std(gdf.loc[gdf["group_name"] == cat, c]) for cat in cats]
            offset = (j - (len(distortion_cols) - 1) / 2) * width
            ax.bar(x + offset, means, width=width, yerr=stds, capsize=4, label=_latex_metric(c))
        ax.axhline(0.0, color="black", linewidth=1.1, linestyle="--")
        ax.set_xticks(x)
        ax.set_xticklabels(cats)
        ax.set_ylabel(r"Distortion term")
        ax.set_title(rf"Primary $\alpha={PRIMARY_ALPHA:.2f}$: distortion terms by group")
        ax.legend(frameon=False)
        _savefig(primary_plot_dir / "primary_alpha_group_distortion_terms.png")

for metric_col, ylabel, fname in [
    ("sigma_delta", _latex_metric("sigma_delta"), "alpha_robustness_sigma_delta.png"),
    ("rms_cov_pooled", _latex_metric("rms_cov_pooled"), "alpha_robustness_rms_cov_pooled.png"),
    ("rms_size_from_groupwise", _latex_metric("rms_size_from_groupwise"), "alpha_robustness_rms_size_from_groupwise.png"),
    ("sigma_lambda", _latex_metric("sigma_lambda"), "alpha_robustness_sigma_lambda.png"),
    ("rms_cov_from_equalized_size", _latex_metric("rms_cov_from_equalized_size"), "alpha_robustness_rms_cov_from_equalized_size.png"),
    ("m_eff_segment_times_sigma_delta", _latex_metric("m_eff_segment_times_sigma_delta"), "alpha_robustness_m_eff_segment_times_sigma_delta.png"),
    ("kappa_eff_times_sigma_lambda", _latex_metric("kappa_eff_times_sigma_lambda"), "alpha_robustness_kappa_eff_times_sigma_lambda.png"),
]:
    _plot_line_with_band(
        df=all_summary_df,
        x_col="alpha",
        y_col=metric_col,
        path=robustness_plot_dir / fname,
        title=rf"Alpha robustness: {_latex_metric(metric_col)}",
        xlabel=r"$\alpha$",
        ylabel=ylabel,
        annotate=True,
    )

if not all_temperature_summary_df.empty:
    temp_df = all_temperature_summary_df.copy()

    for metric_col, ylabel in [
        ("sigma_delta", "sigma_delta"),
        ("rms_cov_pooled", "RMS pooled miscoverage"),
        ("rms_size_from_groupwise", "RMS size distortion"),
        ("sigma_lambda", "sigma_lambda"),
        ("rms_cov_from_equalized_size", "RMS coverage distortion"),
        ("m_eff_segment_times_sigma_delta", "m_eff^seg × sigma_delta"),
        ("kappa_eff_times_sigma_lambda", "kappa_eff × sigma_lambda"),
    ]:
        _plot_line_with_band(
            df=temp_df,
            x_col="temperature_value",
            y_col=metric_col,
            path=temp_plot_dir / f"temperature_sweep_{metric_col}.png",
            title=rf"Controlled sweep at $\alpha={PRIMARY_ALPHA:.2f}$: {_latex_metric(metric_col)}",
            xlabel=_temperature_xlabel(TEMPERATURE_SWEEP_GROUP),
            ylabel=ylabel,
            annotate=True,
        )

    for x_col, y_col, title, ylabel in [
        ("sigma_delta", "rms_cov_pooled", "Controlled sweep: sigma_delta vs pooled RMS miscoverage", "RMS pooled miscoverage"),
        ("sigma_delta", "rms_size_from_groupwise", "Controlled sweep: sigma_delta vs coverage→size distortion", "RMS size distortion"),
        ("sigma_lambda", "rms_cov_from_equalized_size", "Controlled sweep: sigma_lambda vs size→coverage distortion", "RMS coverage distortion"),
        ("m_eff_segment_times_sigma_delta", "rms_cov_pooled", "Controlled sweep: m_eff^seg×sigma_delta vs pooled miscoverage", "RMS pooled miscoverage"),
        ("kappa_eff_times_sigma_lambda", "rms_cov_from_equalized_size", "Controlled sweep: kappa_eff×sigma_lambda vs coverage distortion", "RMS coverage distortion"),
    ]:
        _plot_metric_vs_metric(
            df=temp_df,
            x_col=x_col,
            y_col=y_col,
            path=temp_plot_dir / f"temperature_sweep_{x_col}_vs_{y_col}.png",
            title=title,
            xlabel=_latex_metric(x_col),
            ylabel=ylabel,
            annotate_temp=True,
        )

print("\nRUN_ROOT:", RUN_ROOT)
print("Primary-alpha summary plots:", primary_plot_dir)
print("Alpha-robustness plots:", robustness_plot_dir)
if not all_temperature_summary_df.empty:
    print("Temperature-sweep summary plots:", temp_plot_dir)
print("6-panel manifest:", manifest_path)


In [ ]:
# ============================================================
# Enhanced appendix/reporting exports:
# plot-source CSVs, pretty summary plots, optional
# extra temperature-sweep groups, and score-config manifests.
# Run this AFTER the main plotting cell.
# ============================================================

import json
from pathlib import Path
from matplotlib.ticker import FormatStrFormatter

APPENDIX_EXPORT_DIRNAME = "appendix_reporting_exports"
if "EXTRA_TEMPERATURE_SWEEP_GROUPS" not in globals():
    EXTRA_TEMPERATURE_SWEEP_GROUPS = []


def _plain_numeric_label(name, value):
    try:
        v = float(value)
    except Exception:
        return str(value)
    if name == "alpha":
        return f"{v:.3f}".rstrip("0").rstrip(".") if abs(v) < 1 else f"{v:.4f}"
    if name == "temperature_value":
        return f"{v:.2f}".rstrip("0").rstrip(".")
    return f"{v:.4f}".rstrip("0").rstrip(".")


def _set_yaxis_four_decimals(ax):
    try:
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    except Exception:
        pass


def _export_plot_source_csv(df, x_col, y_col, path, plain_xlabels=False):
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        print(f"[skip] Cannot export CSV for {y_col}: missing data.")
        return

    stats = (
        df.groupby(x_col)[y_col]
        .agg(
            mean=lambda s: _safe_mean(s),
            std=lambda s: _safe_std(s),
            n=lambda s: int(pd.Series(s).dropna().shape[0]),
        )
        .reset_index()
        .sort_values(x_col)
        .reset_index(drop=True)
    )
    if stats.empty:
        print(f"[skip] Empty CSV stats for {y_col}.")
        return

    if plain_xlabels:
        stats["x_label"] = [
            _plain_numeric_label(x_col, float(v)) if pd.notna(v) else str(v)
            for v in stats[x_col]
        ]
    else:
        stats["x_label"] = [
            _annot_label(x_col, float(v)) if pd.notna(v) else str(v)
            for v in stats[x_col]
        ]
    stats["metric"] = y_col
    path.parent.mkdir(parents=True, exist_ok=True)
    stats.to_csv(path, index=False)


def _plot_line_with_band_pretty(
    df,
    x_col,
    y_col,
    path,
    title,
    xlabel,
    ylabel,
    annotate=False,
    annotation_decimals=4,
    plain_xlabels=False,
    yaxis_four_decimals=False,
):
    if df.empty or x_col not in df.columns or y_col not in df.columns:
        print(f"[skip] Cannot plot {y_col}: missing data.")
        return

    stats = _mean_std_by(df, x_col, y_col)
    if stats.empty:
        print(f"[skip] Empty stats for {y_col}.")
        return

    stats = stats.sort_values(x_col).reset_index(drop=True)
    if plain_xlabels:
        labels = [
            _plain_numeric_label(x_col, float(v)) if pd.notna(v) else str(v)
            for v in stats[x_col]
        ]
    else:
        labels = [
            _annot_label(x_col, float(v)) if pd.notna(v) else str(v)
            for v in stats[x_col]
        ]

    y = stats["mean"].to_numpy(dtype=float)
    s = stats["std"].fillna(0.0).to_numpy(dtype=float)
    xpos = np.arange(len(stats), dtype=float)

    fig, ax = plt.subplots(figsize=(3.55, 2.65))
    ax.bar(
        xpos,
        y,
        yerr=s if np.any(s > 0) else None,
        capsize=4 if np.any(s > 0) else 0,
        width=0.72,
        alpha=0.88,
    )
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if yaxis_four_decimals:
        _set_yaxis_four_decimals(ax)
    if annotate:
        for xi, yi in zip(xpos, y):
            ax.annotate(
                f"{yi:.{annotation_decimals}f}",
                (xi, yi),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                fontsize=7.6,
            )
    _savefig(path)


def _requested_temperature_groups():
    ordered = [int(TEMPERATURE_SWEEP_GROUP)]
    for g in EXTRA_TEMPERATURE_SWEEP_GROUPS:
        gi = int(g)
        if gi not in ordered:
            ordered.append(gi)
    return ordered


appendix_root = RUN_ROOT / APPENDIX_EXPORT_DIRNAME
alpha_csv_dir = appendix_root / "alpha_robustness_plot_source_csv"
alpha_plot_dir = appendix_root / "alpha_robustness_pretty_plots"
temp_csv_root = appendix_root / "temperature_plot_source_csv"
temp_plot_root = appendix_root / "temperature_pretty_plots"
config_dir = appendix_root / "score_and_run_config"

for d in [appendix_root, alpha_csv_dir, alpha_plot_dir, temp_csv_root, temp_plot_root, config_dir]:
    d.mkdir(parents=True, exist_ok=True)

requested_groups = _requested_temperature_groups()
score_config = {
    "conformal_score": globals().get("CONFORMAL_SCORE", "unknown"),
    "score_temperature": float(globals().get("SCORE_TEMPERATURE", np.nan)),
    "score_randomize": bool(globals().get("SCORE_RANDOMIZE", False)),
    "score_random_seed": int(globals().get("SCORE_RANDOM_SEED", 0)),
    "raps_lambda": float(globals().get("RAPS_LAMBDA", np.nan)),
    "raps_k_reg": float(globals().get("RAPS_K_REG", np.nan)),
    "saps_lambda": float(globals().get("SAPS_LAMBDA", np.nan)),
    "primary_alpha": float(globals().get("PRIMARY_ALPHA", np.nan)),
    "grid_step": float(globals().get("GRID_STEP", np.nan)),
    "temperature_sweep_group": int(globals().get("TEMPERATURE_SWEEP_GROUP", 0)),
    "requested_temperature_groups": ",".join(str(g) for g in requested_groups),
    "temperature_sweep_values": ",".join(str(float(v)) for v in globals().get("TEMPERATURE_SWEEP_VALUES", [])),
    "seeds": ",".join(str(s) for s in globals().get("SEEDS", [])),
    "run_root": str(RUN_ROOT),
}
with open(config_dir / "score_config.json", "w", encoding="utf-8") as f:
    json.dump(score_config, f, indent=2, ensure_ascii=False)
pd.DataFrame([score_config]).to_csv(config_dir / "score_config.csv", index=False)

alpha_metric_specs = [
    ("sigma_delta", _latex_metric("sigma_delta")),
    ("rms_cov_pooled", _latex_metric("rms_cov_pooled")),
    ("rms_size_from_groupwise", _latex_metric("rms_size_from_groupwise")),
    ("sigma_lambda", _latex_metric("sigma_lambda")),
    ("rms_cov_from_equalized_size", _latex_metric("rms_cov_from_equalized_size")),
    ("m_eff_segment_times_sigma_delta", _latex_metric("m_eff_segment_times_sigma_delta")),
    ("kappa_eff_times_sigma_lambda", _latex_metric("kappa_eff_times_sigma_lambda")),
]

for metric_col, ylabel in alpha_metric_specs:
    _export_plot_source_csv(
        df=all_summary_df,
        x_col="alpha",
        y_col=metric_col,
        path=alpha_csv_dir / f"{metric_col}.csv",
        plain_xlabels=True,
    )
    _plot_line_with_band_pretty(
        df=all_summary_df,
        x_col="alpha",
        y_col=metric_col,
        path=alpha_plot_dir / f"{metric_col}.pdf",
        title=rf"Alpha robustness: {_latex_metric(metric_col)}",
        xlabel=r"$\alpha$",
        ylabel=ylabel,
        annotate=True,
        annotation_decimals=4,
        plain_xlabels=True,
        yaxis_four_decimals=True,
    )

summary_by_alpha_pretty = summary_by_alpha.copy() if "summary_by_alpha" in globals() else pd.DataFrame()
if not summary_by_alpha_pretty.empty:
    summary_by_alpha_pretty.to_csv(appendix_root / "summary_by_alpha_mean_over_seeds_pretty.csv", index=False)

primary_group_summary_pretty = _normalize_group_df(primary_group_summary) if "primary_group_summary" in globals() else pd.DataFrame()
if not primary_group_summary_pretty.empty:
    primary_group_summary_pretty.to_csv(appendix_root / "summary_primary_group_mean_over_seeds_pretty.csv", index=False)

temperature_summary_by_group = {}

if not all_temperature_summary_df.empty and "temperature_group" in all_temperature_summary_df.columns:
    for temp_group, temp_group_df in all_temperature_summary_df.groupby("temperature_group"):
        temperature_summary_by_group[int(temp_group)] = temp_group_df.copy()
elif not all_temperature_summary_df.empty:
    fallback_df = all_temperature_summary_df.copy()
    fallback_df["temperature_group"] = int(TEMPERATURE_SWEEP_GROUP)
    fallback_df["temperature_group_name"] = get_group_name(int(TEMPERATURE_SWEEP_GROUP))
    temperature_summary_by_group[int(TEMPERATURE_SWEEP_GROUP)] = fallback_df

existing_groups = set(temperature_summary_by_group.keys())
seed_dirs = sorted([p for p in RUN_ROOT.glob("seed_*") if p.is_dir()])

for temp_group in requested_groups:
    if temp_group in existing_groups:
        continue

    extra_rows = []
    for seed_dir in seed_dirs:
        try:
            seed = int(str(seed_dir.name).split("_")[-1])
        except Exception:
            seed = str(seed_dir.name)

        probs_cal, y_cal, g_cal, probs_test, y_test, g_test = _load_seed_arrays(seed_dir)
        available_groups = sorted(np.unique(g_cal).astype(int).tolist())
        if int(temp_group) not in available_groups:
            print(f"[skip] group {temp_group} not present in {seed_dir.name}")
            continue

        for temp_value in TEMPERATURE_SWEEP_VALUES:
            temperature_map = {int(g): 1.0 for g in available_groups}
            temperature_map[int(temp_group)] = float(temp_value)

            probs_cal_temp = apply_group_temperature(probs_cal, g_cal, temperature_map, eps=TEMPERATURE_EPS)
            probs_test_temp = apply_group_temperature(probs_test, g_test, temperature_map, eps=TEMPERATURE_EPS)

            temp_result = run_single_alpha_experiment(
                alpha=PRIMARY_ALPHA,
                probs_cal=probs_cal_temp,
                y_cal=y_cal,
                g_cal=g_cal,
                probs_test=probs_test_temp,
                y_test=y_test,
                g_test=g_test,
                plot_dir=None,
                grid_step=GRID_STEP,
                make_plots=False,
                tag_prefix=f"extra_group{int(temp_group)}_temp_{str(temp_value).replace('.', 'p')}_",
            )
            row = dict(temp_result["summary"])
            row["seed"] = seed
            row["temperature_group"] = int(temp_group)
            row["temperature_group_name"] = get_group_name(int(temp_group))
            row["temperature_value"] = float(temp_value)
            row["conformal_score"] = score_config["conformal_score"]
            row["score_temperature"] = score_config["score_temperature"]
            row["score_randomize"] = score_config["score_randomize"]
            row["score_random_seed"] = score_config["score_random_seed"]
            row["saps_lambda"] = score_config["saps_lambda"]
            extra_rows.append(row)

    temperature_summary_by_group[int(temp_group)] = pd.DataFrame(extra_rows)

temperature_summary_mean_by_group = {}
temp_metric_specs = [
    ("sigma_delta", _latex_metric("sigma_delta")),
    ("rms_cov_pooled", _latex_metric("rms_cov_pooled")),
    ("rms_size_from_groupwise", _latex_metric("rms_size_from_groupwise")),
    ("sigma_lambda", _latex_metric("sigma_lambda")),
    ("rms_cov_from_equalized_size", _latex_metric("rms_cov_from_equalized_size")),
    ("m_eff_segment_times_sigma_delta", _latex_metric("m_eff_segment_times_sigma_delta")),
    ("kappa_eff_times_sigma_lambda", _latex_metric("kappa_eff_times_sigma_lambda")),
]

for temp_group in requested_groups:
    temp_df = temperature_summary_by_group.get(int(temp_group), pd.DataFrame()).copy()
    group_csv_dir = temp_csv_root / f"group{int(temp_group)}"
    group_plot_dir = temp_plot_root / f"group{int(temp_group)}"
    group_csv_dir.mkdir(parents=True, exist_ok=True)
    group_plot_dir.mkdir(parents=True, exist_ok=True)

    if temp_df.empty:
        print(f"[skip] No temperature summary available for group {temp_group}.")
        temperature_summary_mean_by_group[int(temp_group)] = pd.DataFrame()
        continue

    temp_df = temp_df.sort_values(["temperature_value", "seed"]).reset_index(drop=True)
    temp_df["conformal_score"] = score_config["conformal_score"]
    temp_df["score_temperature"] = score_config["score_temperature"]
    temp_df["score_randomize"] = score_config["score_randomize"]
    temp_df["score_random_seed"] = score_config["score_random_seed"]
    temp_df["raps_lambda"] = score_config["raps_lambda"]
    temp_df["raps_k_reg"] = score_config["raps_k_reg"]
    temp_df["saps_lambda"] = score_config["saps_lambda"]
    temp_df.to_csv(
        appendix_root / f"all_seeds_temperature_sweep_group{int(temp_group)}.csv",
        index=False,
    )

    mean_cols = [c for c, _ in temp_metric_specs if c in temp_df.columns]
    mean_df = (
        temp_df.groupby("temperature_value", as_index=False)[mean_cols]
        .mean()
        .sort_values("temperature_value")
        .reset_index(drop=True)
    )
    mean_df["temperature_group"] = int(temp_group)
    mean_df["temperature_group_name"] = get_group_name(int(temp_group))
    mean_df["conformal_score"] = score_config["conformal_score"]
    mean_df["score_temperature"] = score_config["score_temperature"]
    mean_df["score_randomize"] = score_config["score_randomize"]
    mean_df["score_random_seed"] = score_config["score_random_seed"]
    mean_df["raps_lambda"] = score_config["raps_lambda"]
    mean_df["raps_k_reg"] = score_config["raps_k_reg"]
    mean_df["saps_lambda"] = score_config["saps_lambda"]
    mean_df.to_csv(
        appendix_root / f"summary_temperature_sweep_group{int(temp_group)}_mean_over_seeds.csv",
        index=False,
    )
    temperature_summary_mean_by_group[int(temp_group)] = mean_df

    for metric_col, ylabel in temp_metric_specs:
        _export_plot_source_csv(
            df=temp_df,
            x_col="temperature_value",
            y_col=metric_col,
            path=group_csv_dir / f"{metric_col}.csv",
            plain_xlabels=True,
        )
        _plot_line_with_band_pretty(
            df=temp_df,
            x_col="temperature_value",
            y_col=metric_col,
            path=group_plot_dir / f"{metric_col}.pdf",
            title=rf"Controlled sweep at $\alpha={PRIMARY_ALPHA:.2f}$: {_latex_metric(metric_col)}",
            xlabel=_temperature_xlabel(int(temp_group)),
            ylabel=ylabel,
            annotate=True,
            annotation_decimals=4,
            plain_xlabels=True,
            yaxis_four_decimals=True,
        )

temperature_summary_mean = temperature_summary_mean_by_group.get(
    int(TEMPERATURE_SWEEP_GROUP),
    pd.DataFrame(),
)

print("\nEnhanced appendix/reporting exports saved to:")
print(appendix_root)
display(summary_by_alpha_pretty)
display(primary_group_summary_pretty)
for temp_group in requested_groups:
    mean_df = temperature_summary_mean_by_group.get(int(temp_group), pd.DataFrame())
    if not mean_df.empty:
        print(f"Temperature sweep mean over seeds: group {temp_group}")
        display(mean_df)



## Suggested next runs

This RAPS notebook is now set up for a cleaner FACET follow-up:

1. **Paper-facing RAPS default**
   - `CONFORMAL_SCORE = "raps"`
   - `SCORE_TEMPERATURE = 0.60`
   - `RAPS_LAMBDA = 0.02`
   - `RAPS_K_REG = 1`

2. **Temperature sweep around the recommended regime**
   - try `SCORE_TEMPERATURE in {0.50, 0.60, 0.70, 0.85, 1.00}`

3. **Then tune the regularization strength**
   - keep `RAPS_K_REG = 1`
   - try `RAPS_LAMBDA in {0.00, 0.02, 0.05, 0.10}`

4. **Only after that, test a slightly larger rank cutoff**
   - `RAPS_K_REG in {1, 2}`

5. **For final stability**
   - `FACET_SPLIT_SIZE_MODE = "published"` or `"large_auto"`
   - `SEEDS = [0, 1, 2]`

The updated notebook also fixes the fixed-temperature bookkeeping, uses a much finer threshold grid, and solves the equalized-size thresholds from the actual attainable set-size support rather than from a coarse grid approximation.
